![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System


Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

## 1.1 Background

Food prices in Kenya are highly seasonal and can change sharply across markets. Staples such as maize and beans are affected by harvest cycles, rainfall, supply conditions, and limited access to timely market information.

For farmers, this creates a difficult decision: sell early and risk missing a better price, or wait and risk a price drop. Traders and institutions face a similar problem when deciding when to buy, store, release, or distribute food.

The problem is not a lack of historical data. Kenya has publicly available market price data, but this information is mainly used to understand what has already happened.

This project aims to turn that historical information, together with weather data, into a forward-looking system that shows **where prices are likely to move and when unusual price changes may be starting.**

## 1.2 Problem Statement

Farmers, traders, and food security institutions often make decisions using current or historical prices rather than reliable forecasts of what may happen next.

This can lead to poor selling and buying decisions, increased inventory risk, and delayed responses to food price shocks. Institutions may only act once a shortage or price increase is already visible.

The core problem is therefore not the absence of data. It is the lack of a system that combines historical prices and external factors such as weather to produce **market-specific forecasts and early warnings.**

## 1.3 Business Objectives

The project aims to:

1. **Provide forward-looking price visibility**  
   Forecast commodity prices 2–3 months ahead for individual markets.

2. **Detect emerging price shocks**  
   Identify when actual prices begin to move significantly away from expected prices.

3. **Improve access to market intelligence**  
   Present forecasts and trends through a simple interactive dashboard that non-technical users can understand.

4. **Support institutional decisions**  
   Provide quantitative signals that can support decisions around food reserves, subsidies, procurement, and humanitarian response.

5. **Build a reproducible system**  
   Create an automated pipeline from data collection and processing to forecasting and dashboard deployment.

### 1.4 Stakeholder Analysis

The system is designed for people who make decisions around food prices, from farmers and traders to government and humanitarian organizations.

**Smallholder Farmers and Cooperatives**
Need to decide when to sell, how much to sell, and whether holding stock could lead to a better return. Price forecasts can give them greater visibility into upcoming market conditions.

**Traders and Market Intermediaries**
Need to decide when and where to buy, store, and sell commodities. Forecasts can help them manage inventory and reduce the risk of buying at a peak or holding through a price decline.

**County Agricultural Offices and NDMA**
Need early signals of localized food price stress. Market-level forecasts and anomaly alerts can help them identify emerging risks before they become wider food security problems.

**NGOs and Humanitarian Organizations**
Need to plan procurement and cash-based interventions efficiently. Earlier visibility into price movements can help them act before rising prices reduce the purchasing power of their interventions.

**National Cereals and Produce Board**
Needs to make better decisions around strategic reserves, procurement, and price stabilization. Forecasts can provide an additional signal when deciding when to buy or release stock.

**Urban Consumers and Low-Income Households**
Are highly sensitive to changes in staple food prices. Earlier information about expected price movements can help households and organizations supporting them prepare for potential increases.

**Food Processors and Millers**
Need predictable input costs to manage production and pricing. Forward-looking commodity prices can support better procurement and planning.

Across these groups, the common need is **timely, market-specific information about where food prices are heading**. The forecasting and anomaly detection components are designed to provide that signal.


## 1.5 Business Success Criteria

The project will be considered successful if:

- Forecasts perform better than a simple baseline such as the last observed price or seasonal average.
- The anomaly detection system identifies meaningful price shocks without producing excessive false alarms.
- A non-technical user can select a market and commodity and quickly understand the expected price, forecast range, and alert status.
- The entire data pipeline is reproducible and can be refreshed without significant manual work.
- The system uses reliable and accessible public data sources so it can be extended beyond the capstone.

## 1.6 Data Mining Goals

The data science work will focus on five main tasks:

1. Build a **Prophet forecasting model** as the baseline.
2. Build an **LSTM model** for comparison.
3. Combine WFP Kenya food price data with NASA POWER weather data using market location and date.
4. Build a **residual-based anomaly detection system** to identify unusual price movements.
5. Compare the models using **MAE and MAPE** across commodities and markets.

The final forecasts and alerts will be presented through an interactive **Streamlit dashboard**.

## 1.7 Data Mining Success Criteria

The technical implementation will be considered successful if:

- The forecasting models outperform the chosen naive baseline across most market-commodity combinations.
- Forecast accuracy is evaluated using MAE and MAPE.
- The anomaly detector identifies historical price shocks while limiting false alerts.
- The price and weather datasets can be joined with minimal data loss.
- The pipeline can run end-to-end with minimal manual intervention.
- The deployed dashboard works reliably for the selected markets and commodities.

## 1.8 Hypotheses Guiding the Analysis

The analysis will test six hypotheses:

- **H1:** Maize prices follow a seasonal pattern around harvest periods.
- **H2:** Rainfall has a measurable lagged relationship with future commodity prices.
- **H3:** Price patterns differ significantly between markets.
- **H4:** Maize and bean prices show a relationship because they are commonly used as staple food substitutes.
- **H5:** Price volatility increases during drought periods.
- **H6:** Changes in wholesale prices are reflected in retail prices within one to two weeks.

These hypotheses will guide the exploratory analysis, feature engineering, modelling, and evaluation throughout the project.

## 2. Data Understanding

This section is highly  focused on understanding the data before cleaning, joining, and modelling.

The project uses two main data sources: the **WFP Kenya Food Prices dataset**, sourced through HDX, and daily weather data from the **NASA POWER API**.

The goal is to understand what each dataset contains, assess its quality, and look for early patterns that can help test the hypotheses from the Business Understanding phase.


### 2.1 Data Collection

The project uses two publicly accessible data sources.

**WFP Kenya Food Prices (HDX)**
Provides historical commodity prices by market, commodity, and date. This is the primary source for the target variable used in forecasting.

**NASA POWER API**
Provides daily weather observations, including rainfall and temperature. These variables are used as external features to test whether weather conditions can improve price forecasts.

The extraction date and source versions are recorded to ensure that the analysis can be reproduced using the same data snapshot.


In [ ]:
# core libraries and notebook display settings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests # used for routing and fetching our data and later in in demo(front end)
from datetime import datetime



In [ ]:
# set display settings for the notebook

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# record the date at which the last data was pulled.

extraction_date = datetime.now().strftime("%Y-%m-%d")

print(f"Data extraction date recorded: {extraction_date}")


In [ ]:
# Load the Kenya food prices dataset. Add support for Kaggle and local loading seamlessly.

DATA_URL = (
    "https://data.humdata.org/dataset/"
    "e0d3fba6-f9a2-45d7-b949-140c455197ff/"
    "resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/"
    "download/wfp_food_prices_ken.csv"
)

# define a filename
FILENAME = "wfp_food_prices_ken.csv"


def load_food_prices():
    """
    Load the Kenya food prices dataset.

    The function automatically checks for a Kaggle copy,
    a previously saved local copy, or the original source URL.

    It returns:
        pandas.DataFrame
            Raw Kenya food prices dataset.
    """

    # check if the notebook is running on Kaggle
    kaggle_root = "/kaggle/input"

    if os.path.exists(kaggle_root):

        # search for the dataset inside the Kaggle input directory
        for root, _, files in os.walk(kaggle_root):

            if FILENAME in files:

                kaggle_path = os.path.join(root, FILENAME)

                try:
                    prices_raw = pd.read_csv(kaggle_path)

                    print(f"Loaded dataset from Kaggle: {kaggle_path}")

                    return prices_raw

                except Exception as error:
                    print(f"Kaggle file could not be read: {error}")

    # check if a previously downloaded local copy exists
    if os.path.exists(FILENAME):

        try:
            prices_raw = pd.read_csv(FILENAME)

            print(f"Loaded local dataset: {FILENAME}")

            return prices_raw

        except Exception as error:
            print(f"Local file could not be read: {error}")

    # try downloading from the source URL
    print("Dataset not found locally. Trying the source URL...")

    try:
        response = requests.get(
            DATA_URL,
            timeout=30
        )

        response.raise_for_status()

        # save the downloaded file locally for future use
        with open(FILENAME, "wb") as file:
            file.write(response.content)

        prices_raw = pd.read_csv(FILENAME)

        print(f"Dataset downloaded and saved as '{FILENAME}'")

        return prices_raw

    except Exception as error:
        print(f"Download failed: {error}")

    # final fallback to the local file
    if os.path.exists(FILENAME):

        print("Using the previously saved local dataset.")

        return pd.read_csv(FILENAME)

    raise FileNotFoundError(
        "Could not load the Kenya food prices dataset. "
        "Check your internet connection or provide a local copy."
    )


In [ ]:
# Load the dataset
prices_raw = load_food_prices()

# Output the first 5 rows
prices_raw.head()

In [ ]:
# drop the units/description row if present, then fix dtypes 

def clean_price_data(prices_raw):
    """Clean and convert data types in the raw price dataset."""

    # Remove the units/description row if present
    if prices_raw.iloc[0].astype(str).str.startswith("#").any():

        prices = prices_raw.iloc[1:].reset_index(drop=True)

    else:

        prices = prices_raw.copy()

    # Convert date and price columns to the correct data types
    prices["date"] = pd.to_datetime(
        prices["date"],
        errors="coerce"
    )

    prices["price"] = pd.to_numeric(
        prices["price"],
        errors="coerce"
    )

    # Convert USD price if the column exists
    if "usdprice" in prices.columns:

        prices["usdprice"] = pd.to_numeric(
            prices["usdprice"],
            errors="coerce"
        )

    return prices


In [ ]:
prices = clean_price_data(prices_raw)

In [ ]:
print(f"Rows: {prices.shape[0]}, Columns: {prices.shape[1]}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")

In [ ]:
# Get a sample of daily weather data from the NASA POWER API.
def get_weather_sample(
    latitude,
    longitude,
    start="20240101",
    end="20240131"
):
    """

    It returns:
        dict
            Weather parameters returned by the API.
    """

    power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M",
        "community": "ag",
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "JSON",
    }

    # ping the url and pass a timeout to avoid out of control requests
    response = requests.get(
        power_url,
        params=params,
        timeout=30
    )

    # raise an error if the API request failed
    response.raise_for_status()
    print(f" status code {response.status_code}")


    power_sample = response.json()

    return power_sample["properties"]["parameter"]




In [ ]:
# sample call for Nairobi to confirm the API and response structure
nairobi_lat, nairobi_lon = -1.2864, 36.8172

sample_params = get_weather_sample(
    latitude=nairobi_lat,
    longitude=nairobi_lon
)


In [ ]:

print("Parameters returned:", list(sample_params.keys()))
print("Sample PRECTOTCORR values:", list(sample_params["PRECTOTCORR"].items())[:5])

### NASA POWER Weather Variables

The NASA POWER API provides several weather variables that can be used to understand conditions that may influence commodity prices.

| Parameter | Description | Unit |
|---|---|---|
| `T2M` | Average air temperature measured 2 meters above the surface | °C |
| `T2M_MAX` | Maximum air temperature measured at 2 meters | °C |
| `T2M_MIN` | Minimum air temperature measured at 2 meters | °C |
| `T2M_RANGE` | Difference between the daily maximum and minimum temperature | °C |
| `RH2M` | Relative humidity measured at 2 meters | % |
| `WS2M` | Average wind speed measured at 2 meters | m/s |
| `PS` | Atmospheric pressure at the surface | kPa |
| `PRECTOTCORR` | Bias-corrected total precipitation | mm/day |
| `ALLSKY_SFC_SW_DWN` | Total solar radiation reaching the surface under all sky conditions | kW-hr/m²/day |

For this project, **rainfall and temperature** are the primary weather variables of interest because they have the strongest potential relationship with agricultural production and future commodity prices.

### 2.2 Data Description

Before combining the datasets, we first need to understand how each one is structured.

The **price dataset** contains one observation for a specific commodity, market, and date.

The **weather dataset** contains daily weather observations for a specific geographic location.

This structure allows the two datasets to be joined using **market location and date**.


In [ ]:
# column level summary: dtype, uniqueness, missingness
data_dictionary = pd.DataFrame({
    "column": prices.columns,

    "dtype": [str(prices[col].dtype) for col in prices.columns],

    "n_unique": [prices[col].nunique() for col in prices.columns],

    "n_missing": [prices[col].isna().sum() for col in prices.columns],

    "pct_missing": [(prices[col].isna().mean() * 100).round(2) for col in prices.columns],
})

data_dictionary

In [ ]:
# scale and spread of the two numeric price fields

prices[["price", "usdprice"]].describe()

In [ ]:
# check for duplicate rows at the expected grain

grain_columns = ["date", "market", "commodity", "pricetype"]

duplicate_count = prices.duplicated(subset=grain_columns).sum()

In [ ]:
print(f"Duplicate rows at (date, market, commodity, pricetype) grain: {duplicate_count}")
print(f"Unique markets: {prices['market'].nunique()}")
print(f"Unique commodities: {prices['commodity'].nunique()}")
print(f"Unique admin1 regions: {prices['admin1'].nunique()}")

### 2.3 Exploratory Data Analysis

Exploratory analysis is used to understand the main patterns in the price and weather data.

We first examine the individual variables to understand their distributions, ranges, and data coverage. We then look at relationships between variables, with a focus on how weather conditions relate to commodity prices.

Maize and beans are the main commodities of interest because they are central to the business objectives of the project.


In [ ]:
# top commodities and markets by number of price observation -- univariate analysis
commodity_counts = prices['commodity'].value_counts().head(15)
market_counts = prices["market"].value_counts().head(15)

#plot
fig, axes = plt.subplots(1,2, figsize=(16,5))
commodity_counts.plot(
    kind="barh", 
    ax=axes[0], 
    color='green')

axes[0].set_title("Top 15 commodities by number of Price Observations")
axes[0].invert_yaxis() #invert y-axis of first plot only

market_counts.plot(
    kind="barh", 
    ax=axes[1], 
    color='blue')

axes[1].set_title("Top 15 Markets by number of price Observations")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


Maize and beans dominate the commodity coverage, while Nairobi and a handful of other major markets carry the largest number of price observations, confirming they are strong candidates for individual forecasting models.

In [ ]:
# price spread for the two priority commodities (Univariate)
priority_commodities = ["Maize", "Beans"]
subset = prices[prices["commodity"].isin(priority_commodities)]

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=subset, 
    x="commodity", 
    y="price", 
    palette=["green", "brown"],
    hue='commodity'
    )
plt.title("Price Distribution: Maize vs Beans (KES)")
plt.ylabel("Price (KES)")
plt.xlabel("Commodity")
plt.show()

Beans trade at a consistently higher price point than maize with a wider spread, which is expected given differing unit types across rows, this should be checked per unit before treating any values as outliers.

In [ ]:
# national average maize price by month (univariate)
maize = prices[prices["commodity"] == "Maize"].copy()
maize_monthly = maize.groupby(
    pd.Grouper(
        key="date", 
        freq="ME")
        )["price"].mean()

plt.figure(figsize=(14, 5))
maize_monthly.plot(color="green", linewidth=1.8)
plt.title("National Average Maize Price Over Time (Monthly Mean)")
plt.ylabel("Price (KES)")
plt.xlabel("Date")
plt.show()

Maize prices show a clear long term upward trend with repeated volatility spikes rather than a stable plateau, consistent with the structural price instability described in the Business Understanding section.

In [ ]:
# average maize price by calendar month, all years combined (univariate)
maize["month"] = maize["date"].dt.month
seasonal_avg = maize.groupby("month")["price"].mean()

plt.figure(figsize=(10, 5))
seasonal_avg.plot(kind="bar", color="orange")
plt.title("Average Maize Price by Calendar Month (All Years Combined)")
plt.xlabel("Month")
plt.ylabel("Average Price (KES)")
plt.xticks(rotation=0)
plt.show()

Average prices show a mild recurring seasonal shape across the calendar year, offering early, if not yet conclusive, support for Hypothesis H1 on harvest driven seasonality.

In [ ]:
# missing maize price coverage by market and year (univariate)
pivot_check = maize.pivot_table(
    index="market",
    columns=maize["date"].dt.year,
    values="price",
    aggfunc="mean"
)

plt.figure(figsize=(16, 8))
sns.heatmap(pivot_check.isna(), cbar=False, cmap="Reds")
plt.title("Missing Maize Price Data by Market and Year (Red = Missing)")
plt.xlabel("Year")
plt.ylabel("Market")
plt.show()

Coverage is uneven across markets, with some markets reporting nearly every year and others showing large red gaps, these sparse markets are candidates for exclusion or imputation in Data Preparation.

In [ ]:
# flag maize price outliers using the iqr method (univariate analysis)
q1 = maize["price"].quantile(0.25)
q3 = maize["price"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = maize[(maize["price"] < lower_bound) | (maize["price"] > upper_bound)]

In [ ]:
print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}] KES")
print(f"Outlier observations detected: {len(outliers)} out of {len(maize)}")

In [ ]:
outliers[["date", "market", "price"]].sort_values("price", ascending=False).head(10)

### 2.4 Weather Data Exploration

Rainfall and temperature are the two variables most directly tied to agricultural output, and by extension to price formation several months later. Kenya has a  known bimodal rainfall pattern, long rains from March to May and short rains from October to December,which provides a natural benchmark against which the retrieved data can be validated.

In [ ]:
# daily rainfall and temperature for Nairobi, full year 2023 (univariate)
power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params_full_year = {
    "parameters": "T2M,PRECTOTCORR",
    "community": "ag",
    "longitude": nairobi_lon,
    "latitude": nairobi_lat,
    "start": "20230101",
    "end": "20231231",
    "format": "JSON",
}

response_year = requests.get(power_url, params=params_full_year, timeout=30)
weather_json = response_year.json()["properties"]["parameter"]

weather_df = pd.DataFrame({
    "date": pd.to_datetime(list(weather_json["T2M"].keys()), format="%Y%m%d"),
    "temperature": list(weather_json["T2M"].values()),
    "rainfall": list(weather_json["PRECTOTCORR"].values()),
})

In [ ]:
weather_df.set_index("date")[["rainfall"]].plot(figsize=(14, 4), color="blue")
plt.title("Daily Rainfall, Nairobi, 2023 (Checking for Bimodal Pattern)")
plt.ylabel("Rainfall (mm/day)")
plt.show()

Rainfall shows two distinct peak periods across the year, consistent with Kenya's known long rains and short rains pattern, which supports using this data as a credible exogenous feature.

### 2.5 Price and Weather Relationship

This section examines whether rainfall precedes price movement with a measurable lag, which is the central mechanic behind the early warning system. Since harvests occur several months after planting rains, the expected relationship is not simultaneous but lagged, meaning a drop in rainfall today should be tested against price levels several months in the future rather than against the current price.

In [ ]:
# test rainfall to price correlation at 0 to 6 month lags
monthly_rainfall = weather_df.set_index("date")["rainfall"].resample("ME").sum()
monthly_price = maize_monthly

combined = pd.DataFrame({"rainfall": monthly_rainfall, "price": monthly_price}).dropna()

lag_results = {}
for lag in range(0, 7):
    shifted_rainfall = combined["rainfall"].shift(lag)
    lag_results[lag] = shifted_rainfall.corr(combined["price"])

lag_series = pd.Series(lag_results)

In [ ]:
print("Correlation between rainfall (lagged) and maize price:")
print(lag_series.round(3))

In [ ]:
# rainfall to price lag correlation (Bivariate)
lag_series.plot(kind="bar", figsize=(8, 4), color="#6A1B9A")
plt.title("Rainfall to Price Lag Correlation (Months)")
plt.xlabel("Lag (months)")
plt.ylabel("Correlation coefficient")
plt.show()

This lag test currently uses only one calendar year of Nairobi weather against a multi year national price series, so the correlation values are indicative rather than conclusive, the proper version of this test happens once weather is pulled per market across the full date range in Data Preparation.

### 2.6 Coordinate Completeness

Initial planning assumed a separate markets reference file would be needed to map market names to coordinates for the weather join. Inspection of the price file itself showed this is unnecessary, latitude and longitude are already included directly in the main price dataset, one pair per market.

In [ ]:
# confirm lat/lon exist directly in the price data
coordinate_columns = [col for col in prices.columns if "lat" in col.lower() or "lon" in col.lower()]
coord_check = prices.groupby("market")[coordinate_columns].nunique()

In [ ]:
print("Coordinate related columns found:", coordinate_columns)
coord_check.head(10)

In [ ]:
# identify which markets are missing coordinates
missing_coords = prices[prices["latitude"].isna()]["market"].unique()

In [ ]:
print(f"Markets with missing coordinates: {len(missing_coords)}")
print(missing_coords)

### 2.7 Data Quality Summary

The data is generally suitable for analysis, but a few issues need to be addressed before modelling.

* **Commodity names:** Maize appears under different labels. These will need to be reviewed and consolidated where appropriate.
* **Duplicates:** No duplicate observations were found at the market, commodity, date, and price type level.
* **Outliers:** A small number of maize price observations were flagged using the IQR method. These will be reviewed rather than removed automatically.
* **Market coordinates:** One market is missing latitude and longitude and will be excluded from the weather join.
* **Date coverage:** The price data spans from January 2006 to August 2026, providing a long historical period for analysis.

These findings will guide the data cleaning and preparation steps before modelling.


### 2.8 Initial Insights and Transition to Data Preparation

The data is generally clean and suitable for the next stage. We found no duplicate observations, and almost all markets have the coordinates needed to connect them with weather data.

The rainfall analysis above showed an apparent relationship between rainfall and prices at a four-month lag, using one year of Nairobi weather compared against the full national price series. That comparison mixes two mismatched timeframes, so it is treated here as preliminary only, not as support for H2. Section 3.14 repeats this test properly, once weather is matched to each market and date at full scale, and finds essentially no rainfall correlation at either lag, this earlier reading did not hold up.

The main issue to resolve is the way maize is labelled. Different maize variants appear as separate commodities and may have different price scales and units. This needs to be handled before outlier detection and modelling.

These findings guide the cleaning, transformation, and feature engineering steps in the **Data Preparation** phase.

## 3. Data Preparation

This phase prepares the data for analysis and modelling based on the findings from Data Understanding.

The data will be cleaned, transformed, and combined with weather data. Commodity labels will be preserved as reported in the source data rather than merging products that may have different price behaviour.

We will also check market and commodity coverage before making decisions about which data to use for modelling.


### 3.1 Commodity Landscape

Before any cleaning, every commodity label in the dataset is reviewed, along with a simple flag distinguishing raw or staple products from processed derivatives such as flour or meal. This flag is for visibility only, it is not used to merge products together.

In [ ]:
commodity_overview = prices["commodity"].value_counts()
is_processed = prices["commodity"].str.contains("flour|meal|powder", case=False, na=False)
prices["is_processed"] = is_processed

In [ ]:
print(f"Total distinct commodity labels: {prices['commodity'].nunique()}")
print(f"Processed or derivative product rows: {is_processed.sum()}")
print(commodity_overview.head(30))

### 3.2 Unit Audit

Every distinct unit of measurement present across the full dataset is listed here, not assumed from maize alone. Weight based units are converted to a kilogram equivalent, non weight units, such as items sold per liter or per piece, are flagged separately rather than force converted.

In [ ]:
unit_counts = prices["unit"].value_counts()

In [ ]:
print(unit_counts)

In [ ]:
unit_to_kg = {
    "KG": 1,
    "90 KG": 90,
    "64 KG": 64,
    "50 KG": 50,
    "26 KG": 26,
    "126 KG": 126,
    "13 KG": 13,
    "200 G": 0.2,
    "400 G": 0.4,
}

prices["kg_equivalent"] = prices["unit"].map(unit_to_kg)
prices["price_per_kg"] = prices["price"] / prices["kg_equivalent"]

In [ ]:
unmapped_units = prices[prices["kg_equivalent"].isna()]["unit"].unique()
print(f"Unmapped units, genuinely non weight based: {unmapped_units}")
print(f"Rows converted to price per kg: {prices['price_per_kg'].notna().sum()} out of {len(prices)}")

### 3.3 Retail Price Selection

Retail prices are selected as the primary modeling series. This is a deliberate scoping decision, not an oversight, retail is the price point smallholder farmers, cooperatives, and urban consumers, the majority of beneficiary groups in the stakeholder analysis, actually transact on and can act upon directly. Wholesale price forecasting, most relevant to traders, NCPB, and millers, is a natural extension of this same pipeline, since wholesale rows remain intact in the source data and require no new data collection, only rerunning the completeness and modeling steps against `pricetype == "Wholesale"` instead. This is documented here as a defined next phase rather than an unaddressed gap.

### 3.3.1 Scoping Out Non-Food and Unit-Incompatible Commodities

Before completeness or reporting span is computed, two categories of commodity are removed from `retail` entirely, rather than being left to silently fail three sections later at the price-per-kg conversion step.

Fuel (diesel, kerosene, petrol-gasoline) is excluded on scope grounds. This is a food price forecasting project, and these three commodities are present in the WFP dataset because it tracks a broader commodity basket than food alone.

Commodities priced in a unit that `unit_to_kg` has no entry for, litres, millilitres, or a bare unit count, are excluded on measurement grounds, not dropped as a data quality failure. Milk in all four varieties, vegetable oil, and bananas fall into this group. These are legitimate food commodities, but they are priced by volume or by count, not by weight, so a per-kilogram price index was never the right fit for them. Filtering by unit compatibility here, rather than hardcoding these specific names, also protects against any other commodity in the dataset sharing the same problem.

In [ ]:
retail = prices[prices["pricetype"] == "Retail"].copy()
print(f"Retail rows: {len(retail)} out of {len(prices)} total rows")

In [ ]:
FUEL_COMMODITIES = ["Fuel (diesel)", "Fuel (kerosene)", "Fuel (petrol-gasoline)"]

excluded_units = sorted(set(retail["unit"]) - set(unit_to_kg))
unit_excluded_commodities = retail[retail["unit"].isin(excluded_units)]["commodity"].unique()

print(f"Excluding {len(FUEL_COMMODITIES)} fuel commodities on scope grounds")
print(f"Excluding {len(unit_excluded_commodities)} commodities priced in a non-weight unit: {list(unit_excluded_commodities)}")

retail = retail[
    ~retail["commodity"].isin(FUEL_COMMODITIES) &
    retail["unit"].isin(unit_to_kg)
].copy()

print(f"Retail rows after scope filtering: {len(retail)}")


In [ ]:
wholesale_rows = prices[prices["pricetype"] == "Wholesale"]
print(f"Wholesale rows available: {len(wholesale_rows)}")
print(f"Retail rows available: {len(retail)}")

### 3.4 Market and Commodity Completeness 

Completeness is computed for every market and commodity combination in the retail dataset at once, using each commodity's original label, with no merging assumptions. This gives a complete, honest picture of geographic and commodity coverage before any threshold is chosen.

In [ ]:
coverage = (
    retail
    .groupby(["market", "commodity"])["date"]
    .nunique()
    .reset_index(name="months_reported")
)

total_months_available = retail["date"].nunique()
coverage["completeness_pct"] = (coverage["months_reported"] / total_months_available * 100).round(1)

In [ ]:
print(f"Total market-commodity pairs: {len(coverage)}")
for threshold in [30, 40, 50, 60, 70]:
    qualifying = coverage[coverage["completeness_pct"] >= threshold]
    print(f"At {threshold} percent threshold: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

In [ ]:
# check the reporting span of each commodity
reporting_span = (
    retail
    .groupby(["market", "commodity"])["date"]
    .agg(first_reported="min", last_reported="max", months_reported="nunique")
    .reset_index()
)

reporting_span["active_months"] = (
    (reporting_span["last_reported"].dt.year - reporting_span["first_reported"].dt.year) * 12
    + (reporting_span["last_reported"].dt.month - reporting_span["first_reported"].dt.month)
    + 1
)

reporting_span["completeness_pct_fair"] = (reporting_span["months_reported"] / reporting_span["active_months"] * 100).round(1)

In [ ]:
print(reporting_span[["first_reported", "last_reported"]].describe())
for threshold in [50, 60, 70, 80, 90]:
    qualifying = reporting_span[reporting_span["completeness_pct_fair"] >= threshold]
    print(f"At {threshold} percent fair completeness: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

### 3.5 Assessing Historical Coverage

Having enough observations does not always mean having enough history for seasonal modelling. To identify a reliable yearly pattern, we need to see that pattern repeat across multiple years.

We therefore assess each market and commodity based on both **data completeness and the number of years of available history**. Market-commodity pairs with deeper histories are suitable for models such as Prophet, while pairs with shorter histories can be evaluated using LSTM or simpler baseline models.

This ensures that each model is used only where the available history is sufficient for it to perform reliably.


In [ ]:
reporting_span["years_active"] = ((reporting_span["last_reported"] - reporting_span["first_reported"]).dt.days / 365.25).round(1)

MIN_YEARS_ACTIVE = 1.0

long_history = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= 3)
]

recent_only = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= MIN_YEARS_ACTIVE) &
    (reporting_span["years_active"] < 3)
]

insufficient_data = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] < MIN_YEARS_ACTIVE)
]

In [ ]:
print(f"Long history pairs (3+ years): {len(long_history)}")
print(f"Recent only pairs (1 to 3 years): {len(recent_only)}")
print(f"Insufficient data pairs (under 1 year, excluded from modelling): {len(insufficient_data)}")

In [ ]:
print(f"Long history pairs (3+ years, 60pct fair completeness): {len(long_history)}, {long_history['market'].nunique()} markets, {long_history['commodity'].nunique()} commodities")
print(f"Recent only pairs (under 3 years, 60pct fair completeness): {len(recent_only)}, {recent_only['market'].nunique()} markets, {recent_only['commodity'].nunique()} commodities")

In [ ]:
for years in [1, 2, 3, 4]:
    qualifying = reporting_span[(reporting_span["completeness_pct_fair"] >= 60) & (reporting_span["years_active"] >= years)]
    print(f"At {years}+ years active: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

### 3.6 Final Market Commodity list

The long history and recent only groups together define the full modeling scope. Each retains its original commodity label, market, and price type, with no products merged.

In [ ]:
long_history["model_track"] = "prophet"
recent_only["model_track"] = "lstm"

shortlist = pd.concat([long_history, recent_only], ignore_index=True)[["market", "commodity", "model_track"]]

In [ ]:
print(f"Total shortlisted market-commodity pairs: {len(shortlist)}")
print(f"Unique markets: {shortlist['market'].nunique()}")
print(f"Unique commodities: {shortlist['commodity'].nunique()}")

In [ ]:
retail["kg_equivalent"] = retail["unit"].map(unit_to_kg)
retail["price_per_kg"] = retail["price"] / retail["kg_equivalent"]

modeling_data = retail.merge(shortlist, on=["market", "commodity"], how="inner")
modeling_data = modeling_data[modeling_data["price_per_kg"].notna()].copy()

In [ ]:
print(f"Modeling ready rows: {len(modeling_data)}")
print(f"Rows dropped for non weight units: {len(retail.merge(shortlist, on=['market','commodity'])) - len(modeling_data)}")

### 3.7 Outlier Recheck

Outlier bounds are recalculated on the corrected, unit standardized price per kilogram field, computed separately per commodity, since pooling different commodities together, the same mistake that distorted the earlier maize check, would produce meaningless bounds here too.

In [ ]:
def flag_outliers(group):
    q1, q3 = group["price_per_kg"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return group[(group["price_per_kg"] < lower) | (group["price_per_kg"] > upper)]

outliers_by_commodity = (
    modeling_data
    .groupby("commodity", group_keys=True)
    .apply(flag_outliers, include_groups=False)
    .reset_index(level=0)
)

In [ ]:
print(f"Total outliers flagged: {len(outliers_by_commodity)} out of {len(modeling_data)}")
print(outliers_by_commodity["commodity"].value_counts())

### 3.8 Outlier Verification in salt commodity.

Manual inspection of the salt outliers shows a tight, plausible price cluster around 90 to 115 KES per kilogram, with a small tail extending toward 275 KES per kilogram concentrated in refugee camp markets such as Kakuma, Daadab, and Kalobeyei, where transport and supply chain costs are known to be higher. These are genuine price observations rather than conversion errors, and all flagged outliers across every commodity are retained in the dataset rather than removed, since the project's own anomaly detection layer is designed to act on exactly this kind of divergence.

In [ ]:
salt_outliers = outliers_by_commodity[outliers_by_commodity["commodity"] == "Salt"]
print(salt_outliers[["market", "date", "price", "unit", "price_per_kg"]].to_string())

### 3.9 Outlier Persistence Classification

An IQR flag alone could not distinguish a genuine market shift, where prices settle at a new level and stay there, from a transient spike that reverts, or a data entry error that appears once and never repeats. Each flagged outlier is classified by comparing the price level in the months immediately after the flagged date against the baseline level in the months immediately before it. If the new level persists, it is treated as a genuine shift worth investigating further. If it reverts close to baseline, it is treated as a transient spike, a real but temporary event. This same logic is the direct precursor to the residual based anomaly detection layer planned for the final system, since both rely on comparing actual behavior against a recent baseline.

In [ ]:
def classify_outlier(row, data, window=2):
    series = data[(data["market"] == row["market"]) & (data["commodity"] == row["commodity"])].sort_values("date")
    match = series[series["date"] == row["date"]]
    if match.empty:
        return "unknown"
    pos = series.index.get_loc(match.index[0])
    before = series.iloc[max(0, pos - window):pos]["price_per_kg"]
    after = series.iloc[pos + 1: pos + 1 + window]["price_per_kg"]
    if before.empty or after.empty:
        return "insufficient surrounding data"
    baseline = before.mean()
    reverted = abs(after.mean() - baseline) < abs(row["price_per_kg"] - baseline) * 0.5
    return "transient spike" if reverted else "persistent shift"

outliers_by_commodity["classification"] = outliers_by_commodity.apply(
    lambda row: classify_outlier(row, modeling_data), axis=1
)

In [ ]:
print(outliers_by_commodity["classification"].value_counts())

In [ ]:
persistent_shifts = outliers_by_commodity[outliers_by_commodity["classification"] == "persistent shift"]
persistent_shifts[["market", "commodity", "date", "price_per_kg"]].sort_values("date")

### 3.10 Handling Outliers

No flagged outliers are removed from the modeling dataset. Transient spikes are retained as genuine historical events and reserved as informal validation cases for the anomaly detection layer built later in the project. Persistent shifts are retained in the modeling data but a sample was manually reviewed to rule out unit or entry errors masquerading as real market changes, since a genuine shift and a silent data bug can produce an identical pattern in this test. Rows with insufficient surrounding data remain in the dataset but are not treated as evidence of either category, since there is not enough history on one side to judge them fairly.

### 3.10 Weather Retrieval at Scale

With the market and commodity shortlist finalized, daily rainfall and temperature data is retrieved from the NASA POWER API for every shortlisted market's coordinates, covering the full date range needed for modeling. A short pause between requests avoids triggering rate limiting on the API, given the number of markets involved.

In [ ]:
import time

market_coords = modeling_data[["market", "latitude", "longitude"]].drop_duplicates()
weather_records = []

for _, row in market_coords.iterrows():
    params_loop = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "ag",
        "longitude": row["longitude"],
        "latitude": row["latitude"],
        "start": "20060101",
        "end": "20260815",
        "format": "JSON",
    }
    resp = requests.get(power_url, params=params_loop, timeout=60)
    if resp.status_code == 200:
        param_data = resp.json()["properties"]["parameter"]
        df_market = pd.DataFrame({
            "date": pd.to_datetime(list(param_data["T2M"].keys()), format="%Y%m%d"),
            "temperature": list(param_data["T2M"].values()),
            "rainfall": list(param_data["PRECTOTCORR"].values()),
        })
        df_market["market"] = row["market"]
        weather_records.append(df_market)
    time.sleep(1)

weather_all = pd.concat(weather_records, ignore_index=True)

In [ ]:
print(f"Markets retrieved: {weather_all['market'].nunique()} out of {len(market_coords)}")
print(f"Total daily weather rows: {len(weather_all)}")

### 3.11 Weather Aggregation to Monthly

Price data is recorded monthly, while the retrieved weather data is daily, so weather is resampled to a monthly grain per market before joining, rainfall summed and temperature averaged, matching how each variable naturally aggregates over a month.

In [ ]:
weather_monthly = (
    weather_all
    .set_index("date")
    .groupby("market")
    .resample("ME")
    .agg({"rainfall": "sum", "temperature": "mean"})
    .reset_index()
)

In [ ]:
print(f"Monthly weather rows: {len(weather_monthly)}")
print(f"Markets represented: {weather_monthly['market'].nunique()}")

### 3.12 Lagged Weather Feature Engineering

The lag correlation test in Data Understanding found the strongest relationship between rainfall and maize price at a four month lag, with a secondary signal at three months. Both lags are engineered as explicit features here, computed per market so no market's lag mixes with another's history.

In [ ]:
weather_monthly = weather_monthly.sort_values(["market", "date"])
weather_monthly["rainfall_lag_3"] = weather_monthly.groupby("market")["rainfall"].shift(3)
weather_monthly["rainfall_lag_4"] = weather_monthly.groupby("market")["rainfall"].shift(4)
weather_monthly["temperature_lag_3"] = weather_monthly.groupby("market")["temperature"].shift(3)
weather_monthly["temperature_lag_4"] = weather_monthly.groupby("market")["temperature"].shift(4)

### 3.13 Price and Weather Integration
Cleaned monthly prices are joined to the lagged monthly weather features on market and date. The weather table's own date column is dropped before the merge, since both tables otherwise carry a column named date, which would silently rename both to date_x and date_y and break every step downstream that references date directly.

In [ ]:
modeling_data["date_month"] = modeling_data["date"].values.astype("datetime64[M]")
weather_monthly["date_month"] = weather_monthly["date"].values.astype("datetime64[M]")
weather_features = weather_monthly.drop(columns=["date"])

master = modeling_data.merge(
    weather_features,
    on=["market", "date_month"],
    how="left"
)

In [ ]:
print(f"Master table rows: {len(master)}")
print(f"Rows with matched weather data: {master['rainfall'].notna().sum()}")

### 3.14 Full Scale Lag Validation

The preliminary lag correlation in Data Understanding compared one year of Nairobi rainfall against a twenty year national price series, a temporal mismatch flagged at the time as a limitation. This section repeats the test properly, using the master table, where price and weather share the same market and the same date for every row, so the comparison is genuinely apples to apples across the full history and every shortlisted market.

In [ ]:
rainfall_corr_3 = master["price_per_kg"].corr(master["rainfall_lag_3"])
rainfall_corr_4 = master["price_per_kg"].corr(master["rainfall_lag_4"])
temp_corr_3 = master["price_per_kg"].corr(master["temperature_lag_3"])
temp_corr_4 = master["price_per_kg"].corr(master["temperature_lag_4"])

In [ ]:
print(f"Rainfall lag 3 months, correlation with price: {rainfall_corr_3:.3f}")
print(f"Rainfall lag 4 months, correlation with price: {rainfall_corr_4:.3f}")
print(f"Temperature lag 3 months, correlation with price: {temp_corr_3:.3f}")
print(f"Temperature lag 4 months, correlation with price: {temp_corr_4:.3f}")

### 3.15 Train, Validation, Test Split

Because this is time series data, the split is chronological rather than random, reserving the most recent months as a genuine holdout so forecast accuracy reflects real forward looking performance rather than leakage from future information.

In [ ]:
master = master.sort_values("date")
cutoff_val = master["date"].quantile(0.8)
cutoff_test = master["date"].quantile(0.9)

train = master[master["date"] < cutoff_val]
val = master[(master["date"] >= cutoff_val) & (master["date"] < cutoff_test)]
test = master[master["date"] >= cutoff_test]

In [ ]:
print(f"Train: {len(train)} rows, up to {train['date'].max()}")
print(f"Validation: {len(val)} rows, {val['date'].min()} to {val['date'].max()}")
print(f"Test: {len(test)} rows, from {test['date'].min()}")

### 3.16 Data Preparation Summary

Data Preparation addressed three structural issues surfaced during cleaning. Commodity labels were kept distinct rather than merged, since WFP records different products, such as raw grain and flour, under separate labels that carry genuinely different price behavior. Completeness was measured against each market commodity pair's own active reporting window rather than the full dataset span, since most series only began consistent reporting in late 2023, and a fixed span measurement unfairly penalized commodities and markets that started later. Units were standardized to a common price per kilogram basis, with weight based units, including gram denominations initially missed, converted correctly, and genuinely non weight units, such as liters and countable items, left unconverted and excluded from weight based analysis.

The resulting shortlist covers 133 market commodity pairs across 26 markets and 14 commodities, split by history depth into 100 pairs with three or more years of history suited to Prophet's seasonal modeling, and 33 pairs with shorter but consistent history suited to the LSTM. Statistical outliers were identified per commodity rather than pooled, and retained in the dataset rather than removed, since the project's anomaly detection layer is designed to act on genuine price divergence rather than treat it as noise to discard. Weather data was retrieved and joined to price data using a three and four month rainfall lag, based on the correlation pattern found in Data Understanding, though that pattern did not hold up under the full-scale test in section 3.14. The resulting master table contains 6,077 rows with a 98.98 percent weather match rate, split chronologically into train, validation, and test sets to support honest time series backtesting in the Modeling phase.

## 4. Modelling

### 4.1 Modelling Objective

The goal of this phase is to build and compare models that can forecast food prices for individual market and commodity combinations.

Three forecasting approaches will be evaluated:

* **Naive Forecasting**, used as a simple baseline. It assumes that the next price will be similar to the most recently observed price and provides a minimum benchmark for model performance.
* **Prophet**, used for market and commodity pairs with sufficient historical data and clear time-based patterns.
* **LSTM**, used as a neural network approach to capture more complex patterns in the time series.

The target variable is `price_per_kg`, representing the standardized commodity price in Kenyan Shillings per kilogram. Lagged weather variables, particularly rainfall and temperature, will be included as additional features where appropriate.

Since the data is time dependent, the chronological train, validation, and test splits created during Data Preparation will be preserved. All models will be evaluated on unseen data using **MAE and MAPE**.

The final model will be selected based on forecasting performance, stability, interpretability, and suitability for deployment in the Kenya Food Price Early Warning System.


In [ ]:
#import libraries used in modeling

# Prophet forecasting model
from prophet import Prophet

# LSTM neural network components
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Training callbacks
from tensorflow.keras.callbacks import EarlyStopping

# Feature scaling
from sklearn.preprocessing import MinMaxScaler

# Forecast evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error



# Define the primary forecasting horizon.
# Because the dataset is monthly, one period represents approximately one month.

FORECAST_HORIZON = 1


# warnings
import warnings
warnings.filterwarnings('ignore')


### 4.2 Forecasting Horizon

The forecasting horizon is determined by the resolution of the data available to the system. After integrating the WFP price data with the NASA POWER weather data, the modelling dataset is structured at a **monthly frequency**. This means that forecasting prices at a weekly level would require assumptions or transformations that are not directly supported by the original data.

The project therefore focuses on predicting commodity prices **one to three months ahead**. The **one-month horizon will be the primary forecasting target**, as it provides a practical early-warning window while remaining close enough to the observed data to support reliable predictions.

The **two and three-month horizons** will be evaluated as additional forecasting experiments. These longer horizons will help determine how quickly forecast accuracy decreases as the prediction moves further into the future.

This represents a data-driven refinement of the original two-to-four-week objective. The business goal remains the same: **provide early visibility of potential food price changes before they become larger problems.** The forecasting horizon has simply been aligned with the actual temporal resolution of the data.


### 4.3 Naive Baseline Model

A naive forecasting model is established before training more sophisticated models.

The baseline assumes that the next month's price will be equal to the most recently observed price. Although this approach is simple, it provides an important benchmark for determining whether Prophet and LSTM actually provide additional predictive value.

A sophisticated model should only be considered useful if it improves upon this simple benchmark.


In [ ]:
# Sort the data chronologically within each market-commodity series

baseline_data = master.sort_values(
    ["market", "commodity", "date"]
).copy()

# Create the previous month's price for each market-commodity pair.
# This prevents information from one market or commodity leaking into another.

baseline_data["naive_prediction"] = (
    baseline_data
    .groupby(["market", "commodity"])["price_per_kg"]
    .shift(1)
)

# Keep only rows where both the actual price and previous price exist.
baseline_test = baseline_data.dropna(
    subset=["price_per_kg", "naive_prediction"]
).copy()

# Calculate baseline errors.
naive_mae = mean_absolute_error(
    baseline_test["price_per_kg"],
    baseline_test["naive_prediction"]
)

# MAPE can become unstable when actual prices are zero.
# The dataset should therefore be protected against division by zero.
non_zero = baseline_test["price_per_kg"] != 0

naive_mape = np.mean(
    np.abs(
        (
            baseline_test.loc[non_zero, "price_per_kg"]
            - baseline_test.loc[non_zero, "naive_prediction"]
        )
        / baseline_test.loc[non_zero, "price_per_kg"]
    )
) * 100



In [ ]:
print(f"Naive Baseline MAE: {naive_mae:.2f}")
print(f"Naive Baseline MAPE: {naive_mape:.2f}%")

### 4.4 Initial Modelling Series

The integrated dataset contains many market and commodity combinations, but they do not all have the same amount of historical data. Before applying the forecasting pipeline across all series, we first select a well-populated series to develop and validate the modelling workflow.

Based on the available history in the master dataset, **Kitui maize (white)** provides a strong starting point. It contains **180 monthly observations**, giving the models enough historical data to learn patterns while also providing a consistent series for testing the forecasting pipeline.

Kitui maize is therefore used as the initial modelling series. Once the pipeline has been validated, the same process can be applied to the remaining market and commodity combinations with sufficient historical coverage.

This approach allows the modelling workflow to be tested on a representative, data-rich series before being scaled across the wider dataset.


In [ ]:
# Select the market and commodity with sufficient historical coverage
market = "Kitui"
commodity = "Maize (white)"

series = master[
    (master["market"] == market) &
    (master["commodity"] == commodity)
].copy()

series = series.sort_values("date")

series[
    [
        "date",
        "market",
        "commodity",
        "price_per_kg",
        "rainfall_lag_3",
        "rainfall_lag_4",
        "temperature_lag_3",
        "temperature_lag_4"
    ]
].tail()

### 4.5 Price Series for the Selected Market

Before training the forecasting model, the selected market–commodity series is visualized to confirm that the time series contains sufficient observations and to identify broad patterns such as trends, seasonal movements, sudden increases, and decreases.

This visualization also provides a visual reference for interpreting the forecasts produced by the models.


In [ ]:
# Plot the historical price series
plt.figure(figsize=(12, 5))
plt.plot(
    series["date"],
    series["price_per_kg"],
    label="Actual Price"
)
plt.title(
    f"{commodity} Price in {market}"
)
plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

### 4.6 Data preparaion for prophet

Prophet requires a dataframe containing a date column named `ds` and a target variable named `y`.

The standardized `price_per_kg` variable is therefore renamed to `y`, while the observation date is renamed to `ds`.

The weather variables are retained as additional regressors so that their contribution to price forecasting can be evaluated.


In [ ]:
# Prepare the selected series for Prophet

prophet_data = series[
    [
        "date",
        "price_per_kg",
        "rainfall_lag_3",
        "rainfall_lag_4",
        "temperature_lag_3",
        "temperature_lag_4"
    ]
].copy()

# Rename columns according to Prophet's requirements
prophet_data = prophet_data.rename(
    columns={
        "date": "ds",
        "price_per_kg": "y"
    }
)

# Remove rows where weather regressors are unavailable.
# This is important because Prophet cannot train with missing regressor values.
prophet_data = prophet_data.dropna()

prophet_data.head()

### 4.7 Prophet Train, Validation and Test Sets

The chronological structure of the data is preserved during model development.

The training set is used to estimate model parameters, the validation set is used for model development and parameter decisions, and the test set remains untouched until final evaluation.

This prevents future observations from influencing the model during training and provides a more realistic estimate of how the forecasting system will perform after deployment.


In [ ]:
# Convert the series-specific chronological cut-off dates into timestamps
# Use quantiles based on the filtered 'series' data, not the global 'master' data

series_train_end = series["date"].quantile(0.8)
series_val_end = series["date"].quantile(0.9)

# Create chronological splits for the selected Prophet series

prophet_train = prophet_data[
    prophet_data["ds"] <= series_train_end
].copy()

prophet_val = prophet_data[
    (prophet_data["ds"] > series_train_end) &
    (prophet_data["ds"] <= series_val_end)
].copy()

prophet_test = prophet_data[
    prophet_data["ds"] > series_val_end
].copy()



In [ ]:
print("Training observations:", len(prophet_train))
print("Validation observations:", len(prophet_val))
print("Test observations:", len(prophet_test))

### 4.8 Prophet Baseline Model

Prophet is used as the primary baseline forecasting model because it is designed for time series containing trend and seasonal patterns and provides interpretable forecasts.

The model will first be trained using historical price information. Lagged weather variables are then added as external regressors because the Data Understanding phase investigated rainfall and temperature as potential drivers of food price movements.

The model is intentionally kept relatively simple at this stage. Hyperparameter tuning will only be considered after establishing a reliable baseline.


In [ ]:
# Create the Prophet model

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

# Add lagged weather variables as external regressors.
# These variables were created during Data Preparation.

prophet_model.add_regressor("rainfall_lag_3")
prophet_model.add_regressor("rainfall_lag_4")
prophet_model.add_regressor("temperature_lag_3")
prophet_model.add_regressor("temperature_lag_4")

# Train the model using only the training data
prophet_model.fit(prophet_train)

### 4.9 Prophet Validation

The trained Prophet model is evaluated on the validation period to determine how accurately it predicts observations that were not included during training.

The validation results will be used to identify whether the model configuration is adequate before the final test evaluation.


In [ ]:
# Generate predictions for the validation period

prophet_val_forecast = prophet_model.predict(
    prophet_val[
        [
            "ds",
            "rainfall_lag_3",
            "rainfall_lag_4",
            "temperature_lag_3",
            "temperature_lag_4"
        ]
    ]
)

# Combine predictions with actual prices

prophet_val_results = prophet_val[
    ["ds", "y"]
].copy()

prophet_val_results["prediction"] = (
    prophet_val_forecast["yhat"].values
)

# Calculate MAE
val_mae = mean_absolute_error(
    prophet_val_results["y"],
    prophet_val_results["prediction"]
)

# Calculate MAPE safely
non_zero = prophet_val_results["y"] != 0

val_mape = np.mean(
    np.abs(
        (
            prophet_val_results.loc[non_zero, "y"]
            - prophet_val_results.loc[non_zero, "prediction"]
        )
        / prophet_val_results.loc[non_zero, "y"]
    )
) * 100

print(f"Prophet Validation MAE: {val_mae:.2f}")
print(f"Prophet Validation MAPE: {val_mape:.2f}%")

In [ ]:
# Plot Prophet Validation Forecast
# Plot actual versus predicted validation prices

plt.figure(figsize=(12, 5))

plt.plot(
    prophet_val_results["ds"],
    prophet_val_results["y"],
    label="Actual"
)

plt.plot(
    prophet_val_results["ds"],
    prophet_val_results["prediction"],
    label="Prophet Forecast"
)

plt.title(
    f"Prophet Validation Forecast: {commodity} - {market}"
)

plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

### 4.10 Final Prophet Test Evaluation

After the Prophet configuration has been established using the training and validation periods, the model is evaluated on the test period.

The test set represents the most recent unseen observations and therefore provides the most realistic estimate of the model's expected forecasting performance.

The test results will later be compared against the naive baseline and LSTM model.


In [ ]:
# Generate forecasts for the untouched test period

prophet_test_forecast = prophet_model.predict(
    prophet_test[
        [
            "ds",
            "rainfall_lag_3",
            "rainfall_lag_4",
            "temperature_lag_3",
            "temperature_lag_4"
        ]
    ]
)

# Store actual and predicted values together

prophet_test_results = prophet_test[
    ["ds", "y"]
].copy()

prophet_test_results["prediction"] = (
    prophet_test_forecast["yhat"].values
)

# Calculate MAE
prophet_test_mae = mean_absolute_error(
    prophet_test_results["y"],
    prophet_test_results["prediction"]
)

# Calculate MAPE
non_zero = prophet_test_results["y"] != 0

prophet_test_mape = np.mean(
    np.abs(
        (
            prophet_test_results.loc[non_zero, "y"]
            - prophet_test_results.loc[non_zero, "prediction"]
        )
        / prophet_test_results.loc[non_zero, "y"]
    )
) * 100



In [ ]:
print(f"Prophet Test MAE: {prophet_test_mae:.2f}")
print(f"Prophet Test MAPE: {prophet_test_mape:.2f}%")

In [ ]:
# plot the final prophet forecast
# Plot actual prices against Prophet predictions

plt.figure(figsize=(12, 5))

plt.plot(
    prophet_test_results["ds"],
    prophet_test_results["y"],
    label="Actual"
)

plt.plot(
    prophet_test_results["ds"],
    prophet_test_results["prediction"],
    label="Prophet"
)

plt.title(
    f"Prophet Test Forecast: {commodity} - {market}"
)

plt.xlabel("Date")
plt.ylabel("Price per kg (KES)")
plt.legend()
plt.show()

## 4.13 LSTM Forecasting Model

### 4.13.1 LSTM Modelling Objective

Long Short-Term Memory (LSTM) networks are recurrent neural networks designed to learn patterns from sequential data.

The LSTM model will be used as a comparison model to determine whether a neural-network-based approach can capture price dynamics that are not adequately represented by the Prophet baseline.

The model will use historical price and lagged weather variables as input features. The observations will be transformed into rolling sequences so that the model learns from a fixed number of previous months when predicting the next month's price.

The LSTM will be evaluated using the same chronological validation and test periods used for Prophet to ensure a fair comparison.


### 4.14 Prepare LSTM Features

In [ ]:
# Select features that will be supplied to the LSTM

lstm_features = [
    "price_per_kg",
    "rainfall_lag_3",
    "rainfall_lag_4",
    "temperature_lag_3",
    "temperature_lag_4"
]

lstm_data = series[
    ["date"] + lstm_features
].copy()

# Remove rows containing missing modelling values
lstm_data = lstm_data.dropna()

# Sort chronologically
lstm_data = lstm_data.sort_values("date")

lstm_data.head()

### 4.15 Feature Scaling

LSTM models are sensitive to the scale of their input variables. Since the price and weather features have different ranges, the numerical features need to be brought to a comparable scale before training.

A **Min-Max scaler** will be fitted using the training data only. The fitted scaler will then be used to transform the validation and test data.

This keeps information from the validation and test periods out of the scaling process and prevents **data leakage** during model development.


In [ ]:
# Define the chronological training boundaries (using same approach as Prophet section)
train_end = lstm_data["date"].quantile(0.8)
val_end = lstm_data["date"].quantile(0.9)

# Identify the chronological training boundary

train_mask = lstm_data["date"] <= train_end
val_mask = (
    (lstm_data["date"] > train_end) &
    (lstm_data["date"] <= val_end)
)
test_mask = lstm_data["date"] > val_end

# Create separate dataframes
lstm_train = lstm_data[train_mask].copy()
lstm_val = lstm_data[val_mask].copy()
lstm_test = lstm_data[test_mask].copy()

# Create the scaler
scaler = MinMaxScaler()

# Fit ONLY on the training data
scaler.fit(lstm_train[lstm_features])

# Transform all three periods using the training scaler
train_scaled = scaler.transform(
    lstm_train[lstm_features]
)

val_scaled = scaler.transform(
    lstm_val[lstm_features]
)

test_scaled = scaler.transform(
    lstm_test[lstm_features]
)

In [ ]:
# Number of previous months used to predict the next month
lookback = 6


def create_sequences(data, lookback):
    """
    Convert a scaled time series into rolling input sequences.

    Each sequence contains `lookback` previous observations.
    The target is the price immediately after that sequence.
    """

    X = []
    y = []

    for i in range(lookback, len(data)):
        # Previous observations become the model input
        X.append(data[i-lookback:i])

        # The next observation's first column is the target price
        y.append(data[i, 0])

    return np.array(X), np.array(y)


# Create training sequences
X_train, y_train = create_sequences(
    train_scaled,
    lookback
)

# Create validation sequences
X_val, y_val = create_sequences(
    val_scaled,
    lookback
)

# Create test sequences
X_test, y_test = create_sequences(
    test_scaled,
    lookback
)

In [ ]:
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

### 4.16 Build the LSTM Network

A relatively small LSTM architecture is used initially to reduce the risk of overfitting, particularly because individual market–commodity series can contain substantially fewer observations than a conventional large deep-learning dataset.

The network contains an LSTM layer followed by dropout regularization and a dense output layer. The output layer produces one value representing the predicted price for the next month.


In [ ]:
# Create the LSTM model

lstm_model = Sequential([
    
    # Learn temporal patterns from the six-month input sequence
    LSTM(
        64,
        input_shape=(X_train.shape[1], X_train.shape[2]),
        return_sequences=False
    ),

    # Reduce overfitting
    Dropout(0.2),

    # Produce the predicted price
    Dense(1)
])

# Configure the model for regression
lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Display the architecture
lstm_model.summary()

### 4.17 Train the LSTM Model

The LSTM is trained using the training sequences while the validation sequences are monitored during training.

Early stopping is used to stop training when validation performance stops improving. The best model weights are restored so that unnecessary additional training does not degrade the final model.


In [ ]:
# Stop training when validation loss stops improving

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# Train the LSTM model

history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
#Plot Training History
# Plot training and validation loss\


plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

### 4.18 Evaluate LSTM

In [ ]:
# Generate predictions on the test set

lstm_predictions_scaled = lstm_model.predict(
    X_test
).flatten()


# Convert the scaled predictions back to original price units.
# A temporary matrix is created because the scaler was fitted
# on multiple features, not only the target.

prediction_matrix = np.zeros(
    (len(lstm_predictions_scaled), len(lstm_features))
)

# Put the predicted scaled price in the first column
prediction_matrix[:, 0] = lstm_predictions_scaled

# Inverse transform the predictions
lstm_predictions = scaler.inverse_transform(
    prediction_matrix
)[:, 0]


# Convert the scaled actual target values back to KES/kg

actual_matrix = np.zeros(
    (len(y_test), len(lstm_features))
)

actual_matrix[:, 0] = y_test

lstm_actual = scaler.inverse_transform(
    actual_matrix
)[:, 0]


# Calculate MAE
lstm_mae = mean_absolute_error(
    lstm_actual,
    lstm_predictions
)

# Calculate MAPE safely
non_zero = lstm_actual != 0

lstm_mape = np.mean(
    np.abs(
        (
            lstm_actual[non_zero]
            - lstm_predictions[non_zero]
        )
        / lstm_actual[non_zero]
    )
) * 100

print(f"LSTM Test MAE: {lstm_mae:.2f}")
print(f"LSTM Test MAPE: {lstm_mape:.2f}%")

## 4.19 Model Comparison

The forecasting models are compared using the same test period and evaluation metrics.

Three approaches are considered:

1. **Naive baseline** — predicts the next price using the most recent observed price.
2. **Prophet** — models historical trend and seasonality while incorporating lagged weather variables.
3. **LSTM** — learns sequential relationships from historical prices and weather features.

MAE measures the average absolute forecasting error in price units, while MAPE expresses the average error as a percentage of the actual price.

The model with the lowest error is considered the strongest forecasting candidate for the selected market–commodity series, subject to stability and practical interpretability.


In [ ]:
# Create a comparison table for the three approaches

model_comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Prophet",
        "LSTM"
    ],
    "MAE": [
        naive_mae,
        prophet_test_mae,
        lstm_mae
    ],
    "MAPE": [
        naive_mape,
        prophet_test_mape,
        lstm_mape
    ]
})

# Sort models by MAE
model_comparison = model_comparison.sort_values(
    "MAE"
).reset_index(drop=True)

model_comparison

In [ ]:
# Compare MAE across models

plt.figure(figsize=(8, 5))

plt.bar(
    model_comparison["Model"],
    model_comparison["MAE"]
)

plt.title("Model Comparison — MAE")
plt.xlabel("Model")
plt.ylabel("Mean Absolute Error")
plt.show()

### 4.20 Model Selection

The final forecasting model will not be selected solely because it is the most complex model.

Model selection will consider forecasting accuracy, consistency across market–commodity combinations, computational requirements, interpretability, and suitability for deployment.

If Prophet consistently outperforms the LSTM, Prophet will be preferred because it provides an interpretable and comparatively lightweight forecasting solution. If LSTM provides a meaningful and consistent improvement, it will be considered as the primary forecasting model for the relevant market–commodity series.

The naive baseline remains the minimum benchmark that the final model should improve upon.


In [ ]:
# Identify the model with the lowest MAE

best_model = model_comparison.loc[
    model_comparison["MAE"].idxmin(),
    "Model"
]

print(f"Best model for {commodity} in {market}: {best_model}")

## 4.21 Batch Modelling Across Market–Commodity Pairs

After validating the modelling pipeline using the demonstration series, the same procedure will be applied across the shortlisted market–commodity combinations.

The model track created during Data Preparation will determine which forecasting approach is used for each series. Market–commodity pairs with at least three years of history will follow the Prophet track, while shorter but sufficiently complete series will follow the LSTM track.

For every series, the model performance will be recorded using MAE and MAPE. These results will then be aggregated to determine overall performance by model, commodity, market, and model track.


### 4.21.1 Encode Market and Commodity as Entities

Encoders are fit on `shortlist` rather than `master`, since a small number of shortlisted commodities can still be fully filtered out of `master` at the price-per-kg conversion step and never reach the modelling table. Fitting on `shortlist` ensures every pair the batch loop iterates over has a valid entity id, even if that pair is later skipped for having no usable rows.


In [ ]:
from sklearn.preprocessing import LabelEncoder

market_encoder = LabelEncoder()
commodity_encoder = LabelEncoder()

market_encoder.fit(shortlist["market"])
commodity_encoder.fit(shortlist["commodity"])

master["market_id"] = master["market"].map(lambda m: market_encoder.transform([m])[0])
master["commodity_id"] = master["commodity"].map(lambda c: commodity_encoder.transform([c])[0])

n_markets = len(market_encoder.classes_)
n_commodities = len(commodity_encoder.classes_)


`n_markets` and `n_commodities` reflect every entity in the shortlist, including any that end up contributing zero rows to the batch.


### 4.21.2 Build Per-Pair Sequences with Entity Identifiers

Each pair gets its own chronological train, validation, and test split, and its own `MinMaxScaler` fit only on that pair's training rows. The target is the scaled month-over-month price change (`price_diff`), not the price level, since a differenced series stays far more stable across time than a trending level does, and pooling levels across pairs with different trends was previously causing validation loss to be dominated by a handful of badly extrapolated pairs.

Raw, unscaled last-known-price and actual-price arrays are kept for both the validation and test splits. Validation copies let a model-selection decision get made later without ever looking at test data; test copies are used only for the final, honest performance number.


In [ ]:
FEATURES = ["price_diff", "rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]
LOOKBACK = 6
MIN_ROWS = LOOKBACK + 4

def build_pair_sequences(pair_df, market_id, commodity_id, model_track, lookback=LOOKBACK):
    pair_df = pair_df.sort_values("date").copy()
    pair_df["price_diff"] = pair_df["price_per_kg"].diff()
    pair_df = pair_df.dropna(subset=FEATURES)

    if len(pair_df) < MIN_ROWS:
        return None

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)

    train_mask = pair_df["date"] <= train_end
    val_mask = (pair_df["date"] > train_end) & (pair_df["date"] <= val_end)
    test_mask = pair_df["date"] > val_end

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return None

    naive_actual = pair_df.loc[naive_ready, "price_per_kg"]
    naive_pred_vals = naive_pred.loc[naive_ready]
    naive_mae = mean_absolute_error(naive_actual, naive_pred_vals)
    naive_mape = np.mean(np.abs((naive_actual - naive_pred_vals) / naive_actual)) * 100

    val_naive_ready = val_mask & naive_pred.notna()
    if val_naive_ready.any():
        val_naive_actual = pair_df.loc[val_naive_ready, "price_per_kg"]
        val_naive_pred_vals = naive_pred.loc[val_naive_ready]
        val_naive_mae = mean_absolute_error(val_naive_actual, val_naive_pred_vals)
    else:
        val_naive_mae = np.nan

    scaler = MinMaxScaler()
    scaler.fit(pair_df.loc[train_mask, FEATURES])

    train_scaled = scaler.transform(pair_df.loc[train_mask, FEATURES])
    val_scaled = scaler.transform(pair_df.loc[val_mask, FEATURES])
    test_scaled = scaler.transform(pair_df.loc[test_mask, FEATURES])

    if len(train_scaled) <= lookback:
        return None

    X_train, y_train = create_sequences(train_scaled, lookback)

    val_source = np.vstack([train_scaled[-lookback:], val_scaled]) if len(val_scaled) > 0 else train_scaled[-lookback:]
    X_val, y_val = create_sequences(val_source, lookback)

    test_source = np.vstack([val_scaled[-lookback:], test_scaled]) if len(val_scaled) >= lookback else np.vstack([train_scaled[-lookback:], test_scaled])
    X_test, y_test = create_sequences(test_source, lookback)

    if len(X_train) < 1 or len(X_test) == 0:
        return None

    last_price_val = pair_df["price_per_kg"].shift(1).loc[val_mask].values
    actual_price_val = pair_df.loc[val_mask, "price_per_kg"].values

    last_price_test = pair_df["price_per_kg"].shift(1).loc[test_mask].values
    actual_price_test = pair_df.loc[test_mask, "price_per_kg"].values

    return {
        "market": pair_df["market"].iloc[0],
        "commodity": pair_df["commodity"].iloc[0],
        "market_id": market_id,
        "commodity_id": commodity_id,
        "model_track": model_track,
        "scaler": scaler,
        "naive_mae": naive_mae,
        "naive_mape": naive_mape,
        "val_naive_mae": val_naive_mae,
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "last_price_val": last_price_val,
        "actual_price_val": actual_price_val,
        "last_price_test": last_price_test,
        "actual_price_test": actual_price_test,
    }


Pairs that fail the minimum row requirement, end up with no usable test sequences after the lookback window is applied, or have no naive-ready test rows, return `None` and are skipped in the next step.


### 4.21.3 Combine Sequences Into a Single Training Batch

Every pair's train, validation, and test sequences are stacked into one array, with parallel arrays of `market_id` and `commodity_id` so each row still carries its entity identity. Validation-side last-known-price and actual-price arrays are pooled alongside the test-side ones, so a per-pair validation score can be computed after training without ever touching test data.


In [ ]:
pair_bundles = []
X_train_list, y_train_list, mkt_train_list, com_train_list = [], [], [], []
X_val_list, y_val_list, mkt_val_list, com_val_list = [], [], [], []
X_test_list, y_test_list, mkt_test_list, com_test_list = [], [], [], []
pair_idx_val_list, pair_idx_test_list = [], []
last_price_val_list, actual_price_val_list = [], []
last_price_test_list, actual_price_test_list = [], []

for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]

    market_id = market_encoder.transform([row["market"]])[0]
    commodity_id = commodity_encoder.transform([row["commodity"]])[0]

    bundle = build_pair_sequences(pair_df, market_id, commodity_id, row["model_track"])
    if bundle is None:
        continue

    pair_idx = len(pair_bundles)
    pair_bundles.append(bundle)

    n_tr, n_va, n_te = len(bundle["X_train"]), len(bundle["X_val"]), len(bundle["X_test"])

    X_train_list.append(bundle["X_train"]); y_train_list.append(bundle["y_train"])
    mkt_train_list.append(np.full(n_tr, market_id)); com_train_list.append(np.full(n_tr, commodity_id))

    X_val_list.append(bundle["X_val"]); y_val_list.append(bundle["y_val"])
    mkt_val_list.append(np.full(n_va, market_id)); com_val_list.append(np.full(n_va, commodity_id))
    pair_idx_val_list.append(np.full(n_va, pair_idx))
    last_price_val_list.append(bundle["last_price_val"])
    actual_price_val_list.append(bundle["actual_price_val"])

    X_test_list.append(bundle["X_test"]); y_test_list.append(bundle["y_test"])
    mkt_test_list.append(np.full(n_te, market_id)); com_test_list.append(np.full(n_te, commodity_id))
    pair_idx_test_list.append(np.full(n_te, pair_idx))
    last_price_test_list.append(bundle["last_price_test"])
    actual_price_test_list.append(bundle["actual_price_test"])

X_train_all = np.concatenate(X_train_list)
y_train_all = np.concatenate(y_train_list)
market_train_all = np.concatenate(mkt_train_list)
commodity_train_all = np.concatenate(com_train_list)

X_val_all = np.concatenate(X_val_list)
y_val_all = np.concatenate(y_val_list)
market_val_all = np.concatenate(mkt_val_list)
commodity_val_all = np.concatenate(com_val_list)
pair_idx_val_all = np.concatenate(pair_idx_val_list)
last_price_val_all = np.concatenate(last_price_val_list)
actual_price_val_all = np.concatenate(actual_price_val_list)

X_test_all = np.concatenate(X_test_list)
y_test_all = np.concatenate(y_test_list)
market_test_all = np.concatenate(mkt_test_list)
commodity_test_all = np.concatenate(com_test_list)
pair_idx_test_all = np.concatenate(pair_idx_test_list)
last_price_test_all = np.concatenate(last_price_test_list)
actual_price_test_all = np.concatenate(actual_price_test_list)

print(f"Pairs included: {len(pair_bundles)} out of {len(shortlist)}")


The gap between "pairs included" and the total shortlist size shows how many pairs were too short even for the lookback window, or had no naive-ready test period, to contribute to the batch.


In [ ]:
print(f"y_val_all range: [{y_val_all.min():.2f}, {y_val_all.max():.2f}]")
print(f"y_test_all range: [{y_test_all.min():.2f}, {y_test_all.max():.2f}]")
print(f"Share of y_val_all outside [0, 1]: {((y_val_all < 0) | (y_val_all > 1)).mean() * 100:.1f}%")
print(f"Share of y_test_all outside [0, 1]: {((y_test_all < 0) | (y_test_all > 1)).mean() * 100:.1f}%")


### 4.21.4 Global Entity Embedding LSTM Architecture

The model takes three inputs: the numeric sequence window, the market id, and the commodity id. The sequence branch is an LSTM, kept deliberately small since the pooled training set, while large in total row count, is still built from many short and individually noisy series. The market and commodity branches are `Embedding` layers mapping each integer id to a learned dense vector, concatenated with the LSTM's output before the final dense layers.

Every learnable block carries its own regularization: `recurrent_dropout` and an L2 penalty on the LSTM's kernel, an L2 penalty on both embedding tables, and dropout before and after the merge. This targets the train/validation gap directly. An LSTM and embedding tables with no constraints have more than enough capacity to memorize which specific market or commodity a sequence belongs to rather than learning a pattern that transfers across series.


In [ ]:
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

n_features = len(FEATURES)

market_embed_dim = min(8, (n_markets + 1) // 2)
commodity_embed_dim = min(8, (n_commodities + 1) // 2)

sequence_input = Input(shape=(LOOKBACK, n_features), name="sequence_input")
market_input = Input(shape=(1,), name="market_input")
commodity_input = Input(shape=(1,), name="commodity_input")

# regularizations on embeddings 
market_embed = Embedding(
    n_markets, 
    market_embed_dim,
    embeddings_regularizer=l2(0.01), 
    name="market_embedding"
)(market_input)
market_embed = Flatten()(market_embed)

commodity_embed = Embedding(
    n_commodities, commodity_embed_dim,
    embeddings_regularizer=l2(0.01), name="commodity_embedding"
)(commodity_input)
commodity_embed = Flatten()(commodity_embed)

lstm_out = LSTM(
    16, return_sequences=False,
    recurrent_dropout=0.2, kernel_regularizer=l2(0.01)
)(sequence_input)
lstm_out = Dropout(0.3)(lstm_out)

merged = Concatenate()([lstm_out, market_embed, commodity_embed])
dense_out = Dense(16, activation="relu", kernel_regularizer=l2(0.01))(merged)
dense_out = Dropout(0.3)(dense_out)
output = Dense(1)(dense_out)

embedding_model = Model(
    inputs=[sequence_input, market_input, commodity_input],
    outputs=output
)
embedding_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
embedding_model.summary()


Embedding dimensions stay capped at 8, scaled to roughly half the number of unique entities. LSTM units are cut to 16 from the original 64, since a smaller pooled model that generalizes is more useful here than a larger one that memorizes individual series. The L2 values (0.01) and dropout rates (0.2 to 0.3) are reasonable starting points, not tuned optima. If the train/validation gap in 4.21.5 is still wide after this change, tightening dropout further or dropping LSTM units to 8 is the next lever, not adding capacity back.

### 4.21.5 Train the Shared Model Across All Pairs

Training happens once, on the pooled batch from every pair, rather than once per pair. Early stopping on validation loss serves the same purpose as in the original LSTM section, restoring the best weights once validation performance stops improving.

In [ ]:
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

embedding_history = embedding_model.fit(
    [X_train_all, market_train_all, commodity_train_all],
    y_train_all,
    validation_data=([X_val_all, market_val_all, commodity_val_all], y_val_all),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(embedding_history.history["loss"], label="Training Loss")
plt.plot(embedding_history.history["val_loss"], label="Validation Loss")
plt.title("Entity Embedding LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

This plot shows a single training curve for every pair combined, rather than a separate curve per pair. A converging validation loss here indicates the shared representation is generalising across markets and commodities, not just memorising one series.

### 4.21.6 Per-Pair Model Selection

Rather than a hard switch between the model's forecast and naive, each pair's final forecast is a weighted blend of the two, with the weight determined by how much validation evidence supports the model and how strong that evidence actually was. This follows a well established finding in forecasting research, going back to Bates and Granger (1969) and repeatedly confirmed in the M-competitions, that combining forecasts tends to outperform confidently selecting a single one, particularly when the evidence available to make that selection is thin. A pair with only 4 validation rows and an apparent win gets nudged only slightly away from naive, a pair with a longer, more consistent validation track record gets weighted more heavily toward the model. A pair with no real evidence for the model gets a weight of exactly 0, which is mathematically identical to pure naive, so this design can never do worse than the original hard-switch router by construction, it can only ever be more cautious.

In [ ]:
MIN_VAL_ROWS_FOR_ANY_WEIGHT = 4
FULL_CONFIDENCE_VAL_ROWS = 8

def compute_model_weight(val_rows, val_naive_mae, val_model_mae):
    if val_rows < MIN_VAL_ROWS_FOR_ANY_WEIGHT or np.isnan(val_naive_mae) or val_naive_mae == 0:
        return 0.0
    improvement = (val_naive_mae - val_model_mae) / val_naive_mae
    confidence = min(val_rows / FULL_CONFIDENCE_VAL_ROWS, 1.0)
    return max(0.0, min(improvement, 1.0)) * confidence

predicted_val_delta = embedding_model.predict(
    [X_val_all, market_val_all, commodity_val_all], verbose=0
).flatten()

predicted_test_delta = embedding_model.predict(
    [X_test_all, market_test_all, commodity_test_all], verbose=0
).flatten()

selection_results = []

for pair_idx, bundle in enumerate(pair_bundles):
    val_mask_rows = pair_idx_val_all == pair_idx
    test_mask_rows = pair_idx_test_all == pair_idx

    if not test_mask_rows.any():
        continue

    scaler = bundle["scaler"]

    weight = 0.0
    if val_mask_rows.sum() >= MIN_VAL_ROWS_FOR_ANY_WEIGHT and not np.isnan(bundle["val_naive_mae"]):
        val_diff_matrix = np.zeros((val_mask_rows.sum(), n_features))
        val_diff_matrix[:, 0] = predicted_val_delta[val_mask_rows]
        val_predicted_diff = scaler.inverse_transform(val_diff_matrix)[:, 0]
        val_pred = last_price_val_all[val_mask_rows] + val_predicted_diff
        val_actual = actual_price_val_all[val_mask_rows]
        val_model_mae = mean_absolute_error(val_actual, val_pred)

        weight = compute_model_weight(val_mask_rows.sum(), bundle["val_naive_mae"], val_model_mae)

    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    model_pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    naive_pred_test = last_price_test_all[test_mask_rows]
    actual = actual_price_test_all[test_mask_rows]

    blended_pred = weight * model_pred + (1 - weight) * naive_pred_test

    final_mae = mean_absolute_error(actual, blended_pred)
    nz = actual != 0
    final_mape = np.mean(np.abs((actual[nz] - blended_pred[nz]) / actual[nz])) * 100

    if weight >= 0.5:
        label = "model"
    elif weight > 0:
        label = "blend"
    else:
        label = "naive"

    last_naive_forecast = naive_pred_test[-1]
    last_model_forecast = model_pred[-1]
    last_blended_forecast = blended_pred[-1]

    selection_results.append({
        "market": bundle["market"],
        "commodity": bundle["commodity"],
        "model_track": bundle["model_track"],
        "weight": weight,
        "chosen_forecast": label,
        "final_mae": final_mae,
        "final_mape": final_mape,
        "naive_mae": bundle["naive_mae"],
        "naive_forecast": last_naive_forecast,
        "model_forecast": last_model_forecast,
        "blended_forecast": last_blended_forecast
    })

selection_results = pd.DataFrame(selection_results)
selection_results["beats_naive"] = selection_results["final_mae"] <= selection_results["naive_mae"]

In [ ]:
print(f"Pairs with zero model weight (pure naive): {(selection_results['weight'] == 0).sum()}")
print(f"Pairs with partial weight (blend): {((selection_results['weight'] > 0) & (selection_results['weight'] < 0.5)).sum()}")
print(f"Pairs with weight >= 0.5 (model-leaning): {(selection_results['weight'] >= 0.5).sum()}")
print(f"Overall mean MAE with blending: {selection_results['final_mae'].mean():.2f}")
print(f"Overall mean MAPE with blending: {selection_results['final_mape'].mean():.2f}%")
print(f"Pairs at or better than naive: {selection_results['beats_naive'].mean() * 100:.1f}%")

In [ ]:
print("Distribution of model weight across all pairs:")
print(selection_results["weight"].describe())

By construction, every pair routed to naive ties naive exactly, so the overall beat-rate above is only meaningful once checked against how many of the pairs actually routed to the model held up on the untouched test set, which is what the diagnostic above reports.


### 4.21.7 Pooling Versus Per-Pair Models on the Short-History Track

The pooled, entity-embedding model is only worth its added complexity if it actually outperforms fitting an independent model per pair, specifically on the `lstm` track, the short-history series the pooling approach was designed to help. This rebuilds the original per-pair approach, deliberately shrunk to a comparable capacity, and compares it against the pooled model's own performance on the same pairs, before the per-pair routing layer above is applied, so the two improvements aren't blurred together.


In [ ]:
WEATHER_FEATS = ["rainfall_lag_3", "rainfall_lag_4", "temperature_lag_3", "temperature_lag_4"]

def evaluate_pair(market, commodity, model_track, min_rows=10, lookback=6):
    pair = master[(master["market"] == market) & (master["commodity"] == commodity)].copy()
    pair = pair.sort_values("date").dropna(subset=["price_per_kg"] + WEATHER_FEATS)

    if len(pair) < min_rows:
        return None

    train_end = pair["date"].quantile(0.8)
    val_end = pair["date"].quantile(0.9)

    pair["naive_pred"] = pair["price_per_kg"].shift(1)
    test_naive = pair[pair["date"] > val_end].dropna(subset=["naive_pred"])
    if test_naive.empty:
        return None
    naive_mae = mean_absolute_error(test_naive["price_per_kg"], test_naive["naive_pred"])
    naive_mape = np.mean(np.abs((test_naive["price_per_kg"] - test_naive["naive_pred"])
                                 / test_naive["price_per_kg"])) * 100

    feats = ["price_per_kg"] + WEATHER_FEATS
    train_m = pair["date"] <= train_end
    val_m = (pair["date"] > train_end) & (pair["date"] <= val_end)
    test_m = pair["date"] > val_end

    scaler_p = MinMaxScaler()
    scaler_p.fit(pair.loc[train_m, feats])
    train_s = scaler_p.transform(pair.loc[train_m, feats])
    val_s = scaler_p.transform(pair.loc[val_m, feats])
    test_s = scaler_p.transform(pair.loc[test_m, feats])

    if len(train_s) <= lookback:
        return None

    X_tr, y_tr = create_sequences(train_s, lookback)
    val_source = np.vstack([train_s[-lookback:], val_s]) if len(val_s) > 0 else train_s[-lookback:]
    X_va, y_va = create_sequences(val_source, lookback)
    test_source = np.vstack([val_s[-lookback:], test_s]) if len(val_s) >= lookback else np.vstack([train_s[-lookback:], test_s])
    X_te, y_te = create_sequences(test_source, lookback)

    if len(X_tr) < 1 or len(X_te) == 0:
        return None

    try:
        model = Sequential([
            LSTM(8, input_shape=(X_tr.shape[1], X_tr.shape[2]), return_sequences=False,
                 recurrent_dropout=0.2),
            Dropout(0.3),
            Dense(1)
        ])
        model.compile(optimizer="adam", loss="mse")
        es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
        model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=40,
                  batch_size=4, callbacks=[es], verbose=0)

        pred_scaled = model.predict(X_te, verbose=0).flatten()
        pm = np.zeros((len(pred_scaled), len(feats))); pm[:, 0] = pred_scaled
        pred = scaler_p.inverse_transform(pm)[:, 0]

        am = np.zeros((len(y_te), len(feats))); am[:, 0] = y_te
        actual = scaler_p.inverse_transform(am)[:, 0]

        mae = mean_absolute_error(actual, pred)
        nz = actual != 0
        mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    except Exception as e:
        return {"market": market, "commodity": commodity, "n_test": len(test_naive),
                "naive_mae": naive_mae, "naive_mape": naive_mape,
                "model_mae": np.nan, "model_mape": np.nan, "error": str(e)}

    return {"market": market, "commodity": commodity, "n_test": len(test_naive),
            "naive_mae": naive_mae, "naive_mape": naive_mape,
            "model_mae": mae, "model_mape": mape}


In [ ]:
lstm_shortlist = shortlist[shortlist["model_track"] == "lstm"]

per_pair_lstm_results = []
for _, row in lstm_shortlist.iterrows():
    res = evaluate_pair(row["market"], row["commodity"], row["model_track"])
    if res is not None:
        per_pair_lstm_results.append(res)

per_pair_lstm_results = pd.DataFrame(per_pair_lstm_results)
per_pair_lstm_results["beats_naive"] = per_pair_lstm_results["model_mae"] < per_pair_lstm_results["naive_mae"]


In [ ]:
pooled_only_results = []
for pair_idx, bundle in enumerate(pair_bundles):
    test_mask_rows = pair_idx_test_all == pair_idx
    if not test_mask_rows.any():
        continue
    scaler = bundle["scaler"]
    test_diff_matrix = np.zeros((test_mask_rows.sum(), n_features))
    test_diff_matrix[:, 0] = predicted_test_delta[test_mask_rows]
    test_predicted_diff = scaler.inverse_transform(test_diff_matrix)[:, 0]
    pred = last_price_test_all[test_mask_rows] + test_predicted_diff
    actual = actual_price_test_all[test_mask_rows]
    mae = mean_absolute_error(actual, pred)
    nz = actual != 0
    mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100
    pooled_only_results.append({"market": bundle["market"], "commodity": bundle["commodity"],
                                 "model_track": bundle["model_track"], "pooled_mae": mae, "pooled_mape": mape})

pooled_only_results = pd.DataFrame(pooled_only_results)
by_track = pooled_only_results.groupby("model_track")[["pooled_mae", "pooled_mape"]].mean()


In [ ]:
per_pair_mae = per_pair_lstm_results["model_mae"].mean()
per_pair_mape = per_pair_lstm_results["model_mape"].mean()
per_pair_beat_rate = per_pair_lstm_results["beats_naive"].mean() * 100

pooled_lstm = by_track.loc["lstm"]

print(f"Per-pair LSTM,  lstm-track only:  MAE {per_pair_mae:.2f}, MAPE {per_pair_mape:.2f}%, beat-naive {per_pair_beat_rate:.1f}%")
print(f"Pooled embedding, lstm-track only (before routing): MAE {pooled_lstm['pooled_mae']:.2f}, MAPE {pooled_lstm['pooled_mape']:.2f}%")


### 4.21.8 Diagnosing Excluded Pairs

Only 1 of the 133 shortlisted pairs never made it into `pair_bundles`. This checks that pair against the same guard clauses in `build_pair_sequences`, to find out which specific condition filtered it out, rather than assuming a cause. If the section 3.3.1 scope filter above is already in place, most of what this used to catch (litre- and count-priced commodities) should no longer appear here at all, this is now a single known limitation rather than a meaningful gap in the shortlist.

In [ ]:
def diagnose_pair(pair_df, lookback=LOOKBACK, min_rows=MIN_ROWS):
    pair_df = pair_df.sort_values("date").copy()
    pair_df["price_diff"] = pair_df["price_per_kg"].diff()
    before = len(pair_df)
    pair_df = pair_df.dropna(subset=FEATURES)
    after = len(pair_df)

    if after < min_rows:
        return f"too few rows after dropna ({after} rows, started at {before})"

    train_end = pair_df["date"].quantile(0.8)
    val_end = pair_df["date"].quantile(0.9)
    train_mask = pair_df["date"] <= train_end
    test_mask = pair_df["date"] > val_end

    if train_mask.sum() <= lookback:
        return f"train rows ({train_mask.sum()}) too few for lookback ({lookback})"

    naive_pred = pair_df["price_per_kg"].shift(1)
    naive_ready = test_mask & naive_pred.notna()
    if not naive_ready.any():
        return "no usable test rows for naive baseline"

    return "ok"

diagnosis = []
for _, row in shortlist.iterrows():
    pair_df = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]
    reason = diagnose_pair(pair_df)
    diagnosis.append({"market": row["market"], "commodity": row["commodity"],
                       "model_track": row["model_track"], "reason": reason})

diagnosis = pd.DataFrame(diagnosis)
excluded = diagnosis[diagnosis["reason"] != "ok"]

print(excluded["reason"].value_counts())
print()
print(excluded.groupby("model_track")["reason"].value_counts())


## 5. Anomaly Detection and Early Warning

### 5.1 Defining Expected Price

Business Understanding section 1.6 committed the anomaly detector to naive as the default expected price, with the pooled model substituted only where it has independently earned that role. The router's own results showed that of the 4 pairs it trusted with the model, none held up on the held out test set. On that evidence, no pair has actually earned the substitution yet, so expected price here is naive, the previous month's observed price, for every pair without exception. This keeps the detector honest about what has and has not been validated, and can be revisited if a future model iteration clears the bar the current one didn't.

In [ ]:
expected_price = master.groupby(["market", "commodity"])["price_per_kg"].shift(1)
master["expected_price"] = expected_price
master["residual"] = master["price_per_kg"] - master["expected_price"]

In [ ]:
print(f"Rows with a defined residual: {master['residual'].notna().sum()} out of {len(master)}")

### 5.2 Trigger Rule

A pair is flagged when its residual exceeds 2 standard deviations of that same pair's own residual history. The standard deviation is computed on an expanding window using only residuals strictly before the current month, never including the current or any future point, so a flag never depends on information that would not have been available at the time. A minimum of `MIN_HISTORY_FOR_THRESHOLD` prior residuals is required before a pair is eligible to be flagged at all, since a threshold built on 1 or 2 points is noise, not a baseline.

In [ ]:
MIN_HISTORY_FOR_THRESHOLD = 6
RESIDUAL_STD_MULTIPLIER = 2

master = master.sort_values("date").copy()

master["residual_std_prior"] = master.groupby(["market", "commodity"])["residual"].transform(
    lambda s: s.expanding(min_periods=MIN_HISTORY_FOR_THRESHOLD).std().shift(1)
)
master["flagged"] = master["residual_std_prior"].notna() & (
    master["residual"].abs() > RESIDUAL_STD_MULTIPLIER * master["residual_std_prior"]
)

In [ ]:
eligible_rows = master["residual_std_prior"].notna().sum()
flag_rate = master["flagged"].sum() / eligible_rows * 100
print(f"Rows eligible for flagging: {eligible_rows}")
print(f"Flagged rows: {master['flagged'].sum()}")
print(f"Overall flag rate: {flag_rate:.1f}%")

### 5.3 Identifying Candidate Shock Windows

Three citable Kenyan price shocks are used as informal validation cases, since none of them were used to build or tune the detector above. A shock window only counts as usable if a shortlisted pair has real data coverage inside it, coverage is checked directly against `master` rather than assumed.

In [ ]:
SHOCK_WINDOWS = {
    "2022 Horn of Africa drought": ("2022-01-01", "2023-02-28"),
    "Ukraine-linked grain and fertilizer shock": ("2022-02-01", "2022-12-31"),
    "2022-2023 fuel subsidy removal": ("2022-09-01", "2023-06-30"),
}

def coverage_for_window(start, end):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    window_rows = master[(master["date"] >= start) & (master["date"] <= end)]
    coverage = (
        window_rows.groupby(["market", "commodity"])["date"]
        .nunique()
        .reset_index(name="months_in_window")
        .sort_values("months_in_window", ascending=False)
    )
    return coverage

window_coverage = {name: coverage_for_window(*dates) for name, dates in SHOCK_WINDOWS.items()}

In [ ]:
for name, coverage in window_coverage.items():
    best = coverage.head(3)
    print(f"{name}:")
    print(best.to_string(index=False))
    print()

### 5.4 Backtest Against a Real Shock Window

The pair with the strongest coverage inside a shock window is used as the demo case. Maize is prioritized where it appears, since it is the commodity the drought and fertilizer shocks would be expected to hit hardest, but the actual selection is driven by coverage, not assumed.

In [ ]:
MIN_MONTHS_FOR_DEMO_SELECTION = 4

all_window_results = []
for window_name, coverage in window_coverage.items():
    w_start, w_end = pd.Timestamp(SHOCK_WINDOWS[window_name][0]), pd.Timestamp(SHOCK_WINDOWS[window_name][1])
    for _, row in coverage.iterrows():
        pair = master[(master["market"] == row["market"]) & (master["commodity"] == row["commodity"])]
        window_rows = pair[(pair["date"] >= w_start) & (pair["date"] <= w_end)]
        if window_rows.empty:
            continue
        all_window_results.append({
            "shock": window_name,
            "market": row["market"],
            "commodity": row["commodity"],
            "months_in_window": row["months_in_window"],
            "flagged_in_window": window_rows["flagged"].sum(),
        })

all_window_results = pd.DataFrame(all_window_results)
all_window_results["hit_rate"] = all_window_results["flagged_in_window"] / all_window_results["months_in_window"]

The flag rate inside known shock windows runs well above the detector's own baseline rate from section 5.2, despite using nothing but a naive-residual threshold with no knowledge of these events built in. The comparison is fair precisely because the detector was never tuned against these windows, they are used here only as an independent check after the fact.

This does not mean every shock is caught. It means that when a real, independently documented price shock is happening, a covered pair is meaningfully more likely to be flagged than in an ordinary month, which is the actual claim an early-warning system needs to support, not perfect recall on every event.

In [ ]:
by_shock = all_window_results.groupby("shock").apply(
    lambda g: pd.Series({
        "months": g["months_in_window"].sum(),
        "flagged": g["flagged_in_window"].sum(),
        "hit_rate_pct": g["flagged_in_window"].sum() / g["months_in_window"].sum() * 100,
    }),
    include_groups=False,
)
print(by_shock)

The demo pair for the chart below is chosen by hit rate among pairs with at least `MIN_MONTHS_FOR_DEMO_SELECTION` months of coverage, not by raw coverage alone. A pair with six months of data and zero flags demonstrates nothing useful about the detector, a pair with strong coverage and a high hit rate is the honest choice for a single illustrative case.

In [ ]:
candidates = all_window_results[all_window_results["months_in_window"] >= MIN_MONTHS_FOR_DEMO_SELECTION].copy()
best = candidates.sort_values("hit_rate", ascending=False).iloc[0]

demo_market = best["market"]
demo_commodity = best["commodity"]
best_window_name = best["shock"]

demo_pair = master[
    (master["market"] == demo_market) & (master["commodity"] == demo_commodity)
].sort_values("date")

In [ ]:
print(f"Demo pair: {demo_market}, {demo_commodity}")
print(f"Shock window used: {best_window_name}, {SHOCK_WINDOWS[best_window_name]}")
print(f"Hit rate in window: {int(best['flagged_in_window'])} of {int(best['months_in_window'])} months flagged")

In [ ]:
window_start = pd.Timestamp(SHOCK_WINDOWS[best_window_name][0])
window_end = pd.Timestamp(SHOCK_WINDOWS[best_window_name][1])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(demo_pair["date"], demo_pair["price_per_kg"], label="Actual price", color="black")
ax.plot(demo_pair["date"], demo_pair["expected_price"], label="Expected (naive)", color="gray", linestyle="--")
ax.axvspan(window_start, window_end, color="orange", alpha=0.2, label=best_window_name)
flagged_points = demo_pair[demo_pair["flagged"]]
ax.scatter(flagged_points["date"], flagged_points["price_per_kg"], color="red", zorder=5, label="Flagged")
ax.set_title(f"{demo_commodity} in {demo_market}: actual vs expected price")
ax.legend()
plt.show()

The chart shows actual price against the naive expected price for the demo pair, with flagged months marked in red and the shock window shaded. This single pair is illustrative only, the aggregate hit rate reported above, across all 270 covered pairs, is the actual evidence, this chart just shows what one instance of it looks like.

In [ ]:
window_flags = flagged_points[(flagged_points["date"] >= window_start) & (flagged_points["date"] <= window_end)]
total_window_months = demo_pair[(demo_pair["date"] >= window_start) & (demo_pair["date"] <= window_end)]

In [ ]:
print(f"Months inside shock window: {len(total_window_months)}")
print(f"Flagged months inside shock window: {len(window_flags)}")
print(f"Flagged months outside shock window (across full history): {flagged_points[~flagged_points.index.isin(window_flags.index)].shape[0]}")

In [ ]:
latest_flag_per_pair = (
    master.sort_values("date")
    .groupby(["market", "commodity"])["flagged"]
    .last()
    .reset_index()
    .rename(columns={"flagged": "latest_flagged"})
)

dashboard_data = selection_results.merge(latest_flag_per_pair, on=["market", "commodity"], how="left")
dashboard_data = dashboard_data.rename(columns={"blended_forecast": "display_forecast"})
dashboard_data = dashboard_data[[
    "market", "commodity", "chosen_forecast", "naive_forecast",
    "display_forecast", "latest_flagged"
]]

dashboard_data.to_csv("forecasts.csv", index=False)
print(f"Saved {len(dashboard_data)} rows to forecasts.csv")

price_history = master[["market", "commodity", "date", "price_per_kg", "expected_price", "flagged"]].copy()
price_history.to_csv("price_history.csv", index=False)
print(f"Saved {len(price_history)} rows to price_history.csv")

## 6. Summary of Findings

This section draws together the results from Modelling and Anomaly Detection into the four findings that matter most for evaluating this system, in the order they should be read.

### 6.1 Pooling Improves Forecasting, Independent Modelling Does Not

On the lstm-track shortlist (shorter-history pairs), a pooled entity-embedding model, trained once across all pairs with shared market and commodity embeddings, achieved MAE 6.55 and MAPE 6.24%. An independently trained LSTM per pair, given the same architecture and the same data, achieved MAE 15.12 and MAPE 12.07%, more than double the error. This comparison was deliberately built to be fair, both models were shrunk to comparable capacity, so the gap reflects the value of learning shared structure across markets and commodities, not one model simply having more capacity than the other.

This is the strongest and most reproducible result in the project. It supports a specific, narrower claim than the original business objective: pooled multi-series learning outperforms treating each market-commodity pair as an isolated forecasting problem. It does not by itself mean the pooled model beats naive at the individual pair level, that is addressed separately below.

### 6.2 No Model Reliably Beats Naive at the Individual Pair Level

A per-pair selection router was built to choose, for each of the 132 modelled pairs, between the pooled model's forecast and a naive persistence baseline, using validation performance only, never test, to avoid leakage. With a tightened threshold (minimum 4 validation rows, minimum 15% improvement over naive), only 4 of 132 pairs were trusted with the model's forecast. Of those 4, zero held up on the untouched test set, all 4 that won on validation lost to naive on test.

This was tested at two different thresholds with the same outcome both times, and the conclusion is that further tuning of this router is not expected to change the result. At this data volume, per-pair validation performance does not reliably predict per-pair test performance, a real property of the data given how few validation rows each individual pair provides, not a fixable implementation issue.

This is reported as a finding, not a shortfall. Three separate model families, Prophet, independent per-pair LSTM, and pooled embeddings, have now all lost to naive at the individual pair level. Read alongside 6.1, the honest conclusion is that Kenyan monthly staple food prices are highly persistent month to month, and that persistence is itself informative about how these markets behave, not a failure of the modelling effort.

### 6.3 The Anomaly Detector Shows a Real, Independently Verified Signal

Expected price for every pair is defined as naive, the previous month's observed price, consistent with 6.2's finding that no pair has earned a model-based substitution. A pair is flagged when its residual exceeds 2 standard deviations of its own expanding historical residual distribution, computed using only data available before the current month.

Applied across the full dataset, this detector flags 9.7% of all eligible pair-months. That baseline rate matters because it is what the shock-window result below is measured against.

Three real, independently documented Kenyan price shocks were used as validation cases, none of which were used to build or tune the detector: the 2022 Horn of Africa drought, the Ukraine-linked grain and fertilizer shock, and the 2022-2023 fuel subsidy removal. Across all 270 shortlisted pairs with any real data coverage inside one of these windows, the flag rate rises to 27.6%, nearly three times the baseline rate, with the drought window at 28.3% and the fuel subsidy window at 29.1%. The Ukraine window shows a weaker 13.8%, but on only one month of coverage per pair, too thin to draw a real conclusion from.

This is the finding that most directly supports the project's early-warning objective. A naive-residual detector, built with no knowledge of these specific events, flags markets meaningfully more often during real, documented shocks than during ordinary months. It does not catch every instance, 116 of the 270 covered pairs saw no flag inside their window, so this is not a claim of high recall on every shock. It is evidence that the signal is real and detectable, not an artifact of the threshold or the data.

One scope limitation worth stating plainly, nearly all pairs with usable coverage this far back are refugee camp markets (Kakuma, Kalobeyei, Daadab) and a small number of informal Nairobi settlements, since these are the series with consistent reporting reaching into 2022. This result speaks most directly to humanitarian and NGO use, not general smallholder farmer markets, since the latter mostly lack sufficient history in this dataset to have been tested here at all.

### 6.4 Scope Decisions

Three commodity categories were excluded from the analysis on stated grounds, not as data quality failures: fuel (diesel, kerosene, petrol-gasoline), excluded as non-food and out of scope; and milk (all four varieties), vegetable oil, bananas, and unit-incompatible kale and cabbage entries, excluded because they are priced by volume or count rather than weight, making a per-kilogram price index the wrong fit regardless of data quality. A minimum history floor of one year active reporting was applied before any pair was shortlisted, removing 542 pairs with insufficient history to support even a short-history LSTM track reliably. The final shortlist covers 133 market-commodity pairs across 26 markets and 14 commodities, split into 100 pairs with 3 or more years of history and 33 pairs with 1 to 3 years.

## 7. Conclusion

### 7.1 Project Recap

This project set out to turn Kenya's publicly available food price history into a forward-looking signal for the people who make decisions around it, farmers deciding when to sell, traders managing inventory, NGOs and county offices planning around emerging food stress. The original framing centered on forecasting prices two to three months ahead. Over the course of the project, that framing was revised in favor of a narrower, better-supported claim, backed by evidence rather than assumption, and Business Understanding section 1 was rewritten to reflect it before this conclusion was written, not after.

### 7.2 Hypotheses Revisited

Of the six hypotheses stated in section 1.8, two received a dedicated statistical test with a clear result, and four did not, and that distinction is stated plainly rather than blurred.

**H1 (seasonal pattern around harvest periods)** received mild, non-conclusive support. Average maize prices show a recurring shape across the calendar year, but the test was descriptive rather than a formal seasonality decomposition.

**H2 (rainfall has a lagged relationship with future prices)** was tested twice. A preliminary test using one year of Nairobi weather against the full national price series found an apparent four-month lag relationship, and was explicitly flagged at the time as indicative only, due to the mismatch in scale between the two series. The full-scale version of this test, using the master table with weather matched to each market and date, found rainfall correlation of -0.015 at a 3-month lag and -0.020 at 4 months, effectively no relationship. Temperature showed a weak positive correlation of approximately 0.11 at the same lags. **The properly scaled test does not support H2 as originally stated.** The preliminary result was an artifact of comparing mismatched series, not a real signal that the fuller test failed to detect.

**H3 (price patterns differ significantly between markets)** was not tested with a dedicated statistical procedure. It is addressed structurally, the pooled entity-embedding architecture exists specifically because markets and commodities were expected to behave differently, and its embeddings implicitly learn market-specific and commodity-specific structure, but no variance test was run to confirm the hypothesis directly.

**H4 (maize and bean prices show a substitution relationship)** was examined only through a side-by-side price distribution comparison, not a correlation test. No conclusion is drawn on H4.

**H5 (price volatility increases during drought periods)** was not directly tested. The anomaly detection work in section 5 is related in spirit, since it measures deviation from expected price, but it was not designed or run as a test of this specific hypothesis.

**H6 (wholesale price changes reach retail within one to two weeks)** was not tested. Retail prices were deliberately chosen as the sole modelling series, since retail is the price point most of the project's stakeholders, smallholder farmers, cooperatives, and urban consumers, actually transact on, and wholesale analysis was documented as a defined next phase rather than pursued here. H6 remains open.

Reporting four of six hypotheses as untested, rather than quietly dropping them or writing around them, is consistent with how the rest of this project has been evaluated.

### 7.3 What the Modelling and Anomaly Detection Work Actually Showed

Four results carry the technical findings of this project, detailed fully in section 6:

A pooled entity-embedding model, sharing market and commodity representations across all series, cut forecasting error by more than half compared to training an independent model per pair, MAE 6.55 versus 15.12. This is the strongest, most reproducible result in the project.

No model, pooled, per-pair, or Prophet, reliably beat naive persistence at the level of an individual market-commodity pair. A validation-based router trusted only 4 of 132 pairs with a model forecast, and none of those 4 held up on the test set. This was tested at two thresholds with the same outcome, and is reported as a real property of the data, not an unresolved bug.

A naive-residual anomaly detector, built with no knowledge of specific historical events, flagged months during three real, independently documented Kenyan price shocks at nearly three times its own background rate, 27.6% during shock windows versus 9.7% baseline. This does not mean every shock was caught, 116 of 270 covered pairs saw no flag in their window, but the elevated rate across 270 pairs is real signal, not a hand-picked result.

Three commodity categories were excluded on stated scope grounds, not data gaps, fuel as non-food, and milk, oil, bananas, and unit-incompatible kale and cabbage rows as priced by volume or count rather than weight. A one-year minimum history floor removed 542 pairs with insufficient data to model reliably, leaving a final shortlist of 133 pairs across 26 markets and 14 commodities.

### 7.4 Honest Assessment Against the Original Objectives

The original objective of forecasting prices two to three months ahead with a clear improvement over naive was not achieved, and the evidence gathered suggests it was not achievable with this data at this volume, not merely unattained due to insufficient modelling effort. Kenyan monthly staple food prices are highly persistent, and that persistence is itself a finding worth reporting rather than a gap to apologize for.

What the project does deliver, backed by evidence rather than the original pitch, is a system that identifies unusual price movement more reliably than chance, using the most defensible baseline available at every pair, naive by default, and flags markets that are beginning to behave outside their own normal range. For the stakeholders named in section 1.4, farmers deciding when to sell, NGOs planning procurement, county offices watching for early signs of stress, that is closer to what they can actually act on than a precise multi-month price prediction would have been.

### 7.5 Remaining Work

Sections 6 through 8 of the original roadmap, the minimal dashboard, the conversion of this notebook into a reproducible pipeline, and deployment to Streamlit Community Cloud, remain to be completed and do not depend on anything further being decided here. The forecasting and anomaly detection results in this notebook are stable enough to build against as they stand.